# ALMA-C11 + CEERS analogues — SED comparison, attenuation, torus inclination

Analysis companion of `analogues_specphot_almac11_rt.ipynb`. Reads ONLY through
`almac11_specphot/almac11_source_map.fits` (built by the staging notebook's Part 6 census), so
reused CEERS trees and new `sed_almac11` trees are transparent.

**Sightline convention.** The dust arms (`dust_on`/`dust_off`) carry 4 sightlines
(`THETA=[0,45,90,135]`), the Nenkova arms a single one (`THETA=[0]`, `findx=0`); the reused
CEERS `nenkova_i90` tree carries 4. Whenever arms are differenced per galaxy
(attenuation, torus residuals) the dust arms are read at `findx=0` and multi-sightline Nenkova
SEDs are averaged; for stacks against observations the sightline mean is used (<0.5% scatter
for these dust-poor hosts either way).

**Observed side.** The CSV carries the CIGALE best-model fluxes (`best.<band>`, mJy) for
19 bands (MegaCam/Suprime u->i, VISTA JHKs, IRAC 1-4, MIPS 24, PACS 100, SPIRE 250, ALMA band 6)
— plotted at band pivot wavelengths. Observed photometry with errors is NOT in the CSV; the
`OBS_PHOT_HOOK` below is the place to point at a staged CIGALE `observations.fits` when copied
to the cluster. AGN properties (L_AGN for the torus-rescaling panel) live in
`ALMAC11_target+control_jwst_AGN_prop.fits` — local only as of 2026-08-04; copy to
`obs_data/almac11/` to enable that panel.

**Parts**: 1 observed side · 2 mock stacks per target · 3 attenuation A_lambda / A_V vs CIGALE
Av_ISM · 4 torus MIR vs inclination (i=90/60/30) · 5 dust masses (sanitized CSV column) ·
6 AGN necessity + obscuration vs observed bands · 7 dust ratios vs attenuation ·
8 host-sightline A_V spread · 9 **consistent A_V** — CIGALE fits of the mock dust_on
photometry in the observed 19 bands (SLURM array; closes the Part 3 estimator mismatch).

**Science questions** (the point of the exercise — analogues were selected in the age–mass
windows where SIMBA predicts dusty sources, dust-to-molecular > 10⁻³ vs controls < 10⁻⁴):
1. Does SIMBA need an **AGN** to reproduce the observed SEDs? → Part 6
2. If so, does the AGN need to be **obscured** (torus i=90 vs 60 vs 30)? → Part 6
3. When the **dust ratios** (dust-to-stellar, dust-to-gas) match, does the **attenuation**
   (Av_ISM) follow? → Part 7, like-for-like in Part 9
4. Does any of this require a particular **inclination** — host sightline (Part 8) or torus
   inclination (Part 6)?


In [ ]:
# ── Part 0 · config + resolvers ──────────────────────────────────────────────
import os, re, glob
from collections import defaultdict

import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import Planck13
import astropy.units as u
from astropy import constants as const

from hyperion.model import ModelOutput

HOME       = '/mnt/home/glorenzon/analize_simba_cgm'
OUTDIR     = os.path.join(HOME, 'output', 'cis100', 'almac11_specphot')
FIGDIR     = os.path.join(OUTDIR, 'figures')
ALMAC11_CSV = os.path.join(HOME, 'ALMAC11_sed_ism_modeling_results.csv')
OBS_PHOT_HOOK = None   # -> staged CIGALE observations.fits for the ALMA-C11 sample (optional)
AGN_PROP_HOOK = os.path.join(HOME, 'obs_data', 'almac11',
                             'ALMAC11_target+control_jwst_AGN_prop.fits')  # optional
os.makedirs(FIGDIR, exist_ok=True)

ARMS = ('dust_on', 'dust_off', 'nenkova_i90', 'nenkova_i60', 'nenkova_i30')
COL = {'no_AGN': 'tab:blue', 'AGN': 'tab:orange', 'obs': 'k',
       'nenkova_i90': '#b30000', 'nenkova_i60': '#e34a33', 'nenkova_i30': '#fc8d59'}

# ── targets: CSV rows + the two CEERS entries ────────────────────────────────
C = Table.read(ALMAC11_CSV, format='csv')
TARGETS = {}
for r in C:
    tid = str(r['ID']).strip()
    TARGETS[tid] = dict(z=float(r['z']), logm=float(r['log(Mstar/Msol)']),
                        age_gyr=10.0 ** float(r['Age(log/yr)']) / 1e9,
                        av_ism=float(r['bayes.attenuation.Av_ISM']),
                        av_ism_err=float(r['bayes.attenuation.Av_ISM_err']),
                        sample='almac11', is_ctrl=tid.lower().startswith('ctrl'),
                        row=dict(zip(r.colnames, r)))
TARGETS['719']  = dict(z=1.463, logm=10.29, age_gyr=2.65, av_ism=np.nan, av_ism_err=np.nan,
                       sample='ceers', is_ctrl=False, row=None)
TARGETS['2962'] = dict(z=1.266, logm=10.68, age_gyr=3.30, av_ism=np.nan, av_ism_err=np.nan,
                       sample='ceers', is_ctrl=False, row=None)
_tid_safe = lambda tid: re.sub(r'[^A-Za-z0-9_-]', '', tid)

# ── observed-marker style: the `ctrl*` CSV rows are the dust-poor control
# sample (logDGR < -4) and are drawn hollow wherever the observed Av / dust
# mass appears; the dusty ALMA-C11 targets stay filled. ──────────────────────
def obs_mstyle(tid, color='k'):
    return (dict(mfc='none', mec=color, mew=1.3) if TARGETS[tid]['is_ctrl']
            else dict(mfc=color, mec=color, mew=0.8))

def obs_proxy(color='k'):
    return [Line2D([], [], ls='none', marker='s', ms=6, mfc=color, mec=color,
                   label='observed (dusty target)'),
            Line2D([], [], ls='none', marker='s', ms=6, mfc='none', mec=color,
                   mew=1.3, label='observed (control)')]

N_CTRL = sum(T['is_ctrl'] for T in TARGETS.values())

# ── selection tables + source map ────────────────────────────────────────────
SELECTED = {tid: Table.read(os.path.join(OUTDIR, f'rt_selection_{_tid_safe(tid)}.fits'))
            for tid in TARGETS
            if os.path.exists(os.path.join(OUTDIR, f'rt_selection_{_tid_safe(tid)}.fits'))}
with fits.open(os.path.join(OUTDIR, 'rt_union.fits')) as _h:
    UNION = Table(_h['UNION'].data)
    MEMBERS = Table(_h['MEMBERS'].data)
SMAP = Table.read(os.path.join(OUTDIR, 'almac11_source_map.fits'))
for _t in (UNION, MEMBERS, SMAP):          # FITS strings load as bytes; decode once
    _t.convert_bytestring_to_unicode()

_smap = {}
for r in SMAP:
    if bool(r['COMPLETE']):
        _smap[(str(r['ARM']), int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT']))] = str(r['RTOUT_PATH'])

def rtout_path(arm, snap, gid):
    return _smap.get((arm, int(snap), int(gid)))   # None if RT not complete

_as_str = lambda col: np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x)
                                for x in col])

# ── SED reader (largest aperture; sightline handling per convention above) ───
_SED_CACHE = {}
def read_sed(arm, snap, gid, sightline='mean'):
    # -> (wav_rest_um, nuLnu_erg_s) or None if the RT is not complete.
    key = (arm, int(snap), int(gid), sightline)
    if key not in _SED_CACHE:
        p = rtout_path(arm, snap, gid)
        if p is None:
            return None
        m = ModelOutput(p)
        sed = m.get_sed(inclination='all', aperture=-1)
        wav = np.asarray(sed.wav, float)
        val = np.atleast_2d(np.asarray(sed.val, float))
        val = val[0] if (sightline == 'findx0' or val.shape[0] == 1) else val.mean(axis=0)
        s = np.argsort(wav)
        _SED_CACHE[key] = (wav[s], val[s])
    return _SED_CACHE[key]

GRID = np.geomspace(0.08, 1500.0, 400)   # rest-frame micron

def on_grid(wl, nuLnu):
    with np.errstate(invalid='ignore', divide='ignore'):
        return 10 ** np.interp(np.log10(GRID), np.log10(wl), np.log10(nuLnu),
                               left=np.nan, right=np.nan)

def attenuation_curve(snap, gid):
    # A_lambda on GRID from the dust_on/dust_off findx0 pair (None if RT missing)
    on = read_sed('dust_on', snap, gid, sightline='findx0')
    off = read_sed('dust_off', snap, gid, sightline='findx0')
    if on is None or off is None:
        return None
    with np.errstate(invalid='ignore', divide='ignore'):
        return -2.5 * np.log10(on_grid(*on) / on_grid(*off))

_jV = int(np.argmin(np.abs(GRID - 0.55)))    # V-band index on GRID

def obs_fnu_factor(z):
    # (lambda_obs [um], factor nuLnu[erg/s] -> Fnu [mJy]) at redshift z
    wav_obs = GRID * (1.0 + z)
    dl = Planck13.luminosity_distance(z).to(u.cm).value
    nu_obs = (const.c / (wav_obs * u.micron)).to(u.Hz).value
    return wav_obs, 1.0 / (4.0 * np.pi * dl**2 * nu_obs) / 1e-26

print(f'{len(TARGETS)} targets ({N_CTRL} controls, hollow markers) | '
      f'{len(UNION)} union galaxies | '
      f'{len(SMAP)} source-map rows ({int(np.asarray(SMAP["COMPLETE"]).sum())} complete)')
for arm in ARMS:
    m_ = np.asarray(SMAP['ARM']) == arm
    #print(f'  {arm:12s} {int(np.asarray(SMAP["COMPLETE"])[m_].sum()):4d} / {int(m_.sum()):4d} complete')


## Part 1 · observed side — CIGALE best-model fluxes at band pivots

`best.<band>` are model fluxes in mJy (no errors). Pivot wavelengths below are observed-frame
microns; replace with the exact CIGALE filter pivots if a filters directory is staged.


In [ ]:
# ── Part 1 · observed photometry (CIGALE best-model fluxes) ──────────────────
BAND_PIVOT_UM = {              # observed-frame pivot wavelength [um]
    'best.MCam_u': 0.381,  'best.subaru.suprime.IB427': 0.426, 'best.SUBARU_B': 0.446,
    'best.subaru.suprime.IB464': 0.464, 'best.MCam_g': 0.487, 'best.subaru.suprime.V': 0.548,
    'best.subaru.suprime.r': 0.629, 'best.SUBARU_i': 0.768,
    'best.vista.vircam.J': 1.254, 'best.vista.vircam.H': 1.646, 'best.vista.vircam.Ks': 2.149,
    'best.IRAC1': 3.557, 'best.IRAC2': 4.504, 'best.IRAC3': 5.738, 'best.IRAC4': 7.927,
    'best.MIPS1': 23.68, 'best.PACS_green': 100.0, 'best.PSW': 250.0, 'best.ALMA6': 1250.0,
}

OBS = {}
for tid, T in TARGETS.items():
    if T['row'] is None:
        continue                       # CEERS: observed side lives in the cis100 notebook
    wl = np.array([BAND_PIVOT_UM[b] for b in BAND_PIVOT_UM])
    f_ = np.array([float(T['row'][b]) for b in BAND_PIVOT_UM])
    OBS[tid] = dict(wl=wl, fnu=f_)     # mJy, observed frame

# optional: real observed photometry with errors, once staged on the cluster
if OBS_PHOT_HOOK and os.path.exists(OBS_PHOT_HOOK):
    print('observations.fits hook found — wire the per-band errors in here')
print(f'observed side ready for {len(OBS)} ALMA-C11 targets '
      f'({len(BAND_PIVOT_UM)} bands, 0.38 um - 1.25 mm observed)')


## Part 2 · mock stacks per target — dust_on vs observed

Median + 16-84% band of the member analogues' `dust_on` SEDs, placed at the target redshift,
against the CIGALE best-model fluxes. AGN/non-AGN split via the per-target `AGN_CLASS`.


In [ ]:
# ── Part 2 · per-target SED stacks ───────────────────────────────────────────
def stack_curves(rows, arm, sightline='mean'):
    curves = []
    for r in rows:
        sed = read_sed(arm, r['SNAPSHOT'], r['GROUPID_SNAPSHOT'], sightline=sightline)
        if sed is not None:
            curves.append(on_grid(*sed))
    C_ = np.asarray(curves)
    if C_.size == 0:
        return None
    return dict(med=np.nanmedian(C_, axis=0), lo=np.nanpercentile(C_, 16, axis=0),
                hi=np.nanpercentile(C_, 84, axis=0), n=len(C_), curves=C_)

for tid in [t for t in TARGETS if t in SELECTED and t in OBS]:
    A = SELECTED[tid]
    A_cls = _as_str(A['AGN_CLASS'])
    z = TARGETS[tid]['z']
    WOBS, FAC = obs_fnu_factor(z)
    fig, ax = plt.subplots(figsize=(8.5, 6))
    for cls in ('no_AGN', 'AGN'):
        S = stack_curves(A[A_cls == cls], 'dust_on')
        if S is None:
            continue
        ax.fill_between(WOBS, S['lo'] * FAC, S['hi'] * FAC, color=COL[cls], alpha=0.25, lw=0)
        ax.plot(WOBS, S['med'] * FAC, color=COL[cls], lw=2,
                label=f"SIMBA {'AGN host' if cls == 'AGN' else 'non-AGN host'} (N={S['n']})")
    ax.plot(OBS[tid]['wl'], OBS[tid]['fnu'], 'o', ms=6, mfc='w', mec=COL['obs'],
            label='CIGALE best-model fluxes', zorder=5)
    ax.set(xscale='log', yscale='log', xlim=(0.2, 2000),
           xlabel=r'observed-frame wavelength [$\mu$m]', ylabel=r'$F_\nu$ [mJy]',
           title=f"{tid} (z={z:.3f}) vs SIMBA analogues — dust_on")
    fin = OBS[tid]['fnu'][OBS[tid]['fnu'] > 0]
    ax.set_ylim(fin.min() / 3e2, fin.max() * 3e2)
    ax.grid(alpha=0.2, which='both', lw=0.5)
    ax.legend(fontsize=9, frameon=False, loc='lower center')
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, f'sed_stack_{_tid_safe(tid)}.png'),
                dpi=180, bbox_inches='tight')
    plt.show()


## Part 4 · torus MIR strength vs CLUMPY inclination

AGN hosts only. Per galaxy the torus contribution is the residual `nenkova_iXX − dust_on`
(same galaxy, same dust, only the injected template differs; dust_on at `findx=0`). Median
residuals for i=90/60/30, plus the ratio-to-dust_on version. Optional: rescale each residual by
`L_AGN,obs / L_AGN,injected` (needs the AGN-prop FITS staged — see header).


In [ ]:
# ── Part 4 · torus residual vs inclination ───────────────────────────────────
_agn_rows = UNION[np.asarray(UNION['AGN_ANY'], bool)]
print(f'AGN hosts with torus arms: {len(_agn_rows)}')

RES = {}
for arm in ('nenkova_i90', 'nenkova_i60', 'nenkova_i30'):
    res, ratio = [], []
    for r in _agn_rows:
        snap, gid = int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])
        on = read_sed('dust_on', snap, gid, sightline='findx0')
        nk = read_sed(arm, snap, gid, sightline='mean')   # mean handles the reused 4-sightline i90
        if on is None or nk is None:
            continue
        g_on, g_nk = on_grid(*on), on_grid(*nk)
        res.append(g_nk - g_on)
        with np.errstate(invalid='ignore', divide='ignore'):
            ratio.append(g_nk / g_on)
    if res:
        RES[arm] = dict(res=np.asarray(res), ratio=np.asarray(ratio), n=len(res))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for arm, D in RES.items():
    axes[0].plot(GRID, np.nanmedian(D['res'], axis=0), color=COL[arm], lw=2,
                 label=f"{arm} (N={D['n']})")
    axes[1].plot(GRID, np.nanmedian(D['ratio'], axis=0), color=COL[arm], lw=2)
axes[0].set(xscale='log', yscale='log', xlim=(0.1, 500), ylabel=r'$\nu L_\nu$ [erg s$^{-1}$]',
            xlabel=r'rest-frame wavelength [$\mu$m]',
            title='median torus residual (nenkova $-$ dust_on)')
axes[1].axhline(1, color='0.5', lw=0.8)
axes[1].set(xscale='log', yscale='log', xlabel=r'rest-frame wavelength [$\mu$m]',
            ylabel='nenkova / dust_on', title='median boost over the host')
axes[0].legend(fontsize=9, frameon=False)
for ax in axes:
    ax.grid(alpha=0.2, which='both', lw=0.5)
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'torus_vs_inclination.png'), dpi=180)
plt.show()

# optional L_AGN rescaling: residual * (L_AGN_obs / L_AGN_injected), L_AGN_injected = LBOL (0.1 Mdot c^2)
if os.path.exists(AGN_PROP_HOOK):
    print('AGN-prop FITS found — add the matched-L_AGN panel here')
else:
    print(f'AGN-prop FITS not staged ({AGN_PROP_HOOK}) — matched-L_AGN panel skipped')


## Part 5 · dust masses — sanitized CSV column vs simulated analogues

The CSV `M_dust [Msun]` column has **mixed units across rows** (some raw Msun, some scaled).
Adjudicate row-by-row against `Mdust_sc` (+err) and `Mdust_mbb`; adopt `Mdust_sc` where the raw
column is inconsistent, and print the adjudication so nothing is silent.


In [ ]:
# ── Part 5 · dust-mass comparison ────────────────────────────────────────────
print(f"{'tid':8s} {'M_dust[raw]':>12s} {'Mdust_sc':>12s} {'Mdust_mbb':>12s} {'adopted logMd':>13s}")
LOGMD_OBS = {}
for tid, T in TARGETS.items():
    if T['row'] is None:
        continue
    raw, sc, mbb = (float(T['row'].get(k_, np.nan)) for k_ in
                    ('M_dust [Msun]', 'Mdust_sc', 'Mdust_mbb'))
    # raw is trustworthy only when it agrees with Mdust_sc within a factor of a few
    adopted = raw if (np.isfinite(raw) and np.isfinite(sc) and 0.3 < raw / sc < 3) else sc
    LOGMD_OBS[tid] = np.log10(adopted) if np.isfinite(adopted) and adopted > 0 else np.nan
    flag = '' if adopted is raw else '   <- raw inconsistent, using Mdust_sc'
    print(f'{tid:8s} {raw:12.3e} {sc:12.3e} {mbb:12.3e} {LOGMD_OBS[tid]:13.2f}{flag}')

_tids = [t for t in LOGMD_OBS if t in SELECTED]
fig, ax = plt.subplots(figsize=(9, 5))
for x, tid in enumerate(_tids):
    y = np.asarray(SELECTED[tid]['LOG_MDUST'], float)
    y = y[np.isfinite(y)]
    ax.plot(np.full(y.size, x) + np.random.uniform(-0.15, 0.15, y.size), y, 'o', ms=3,
            color='tab:blue', alpha=0.4, mec='none')
    if y.size:
        ax.hlines(np.median(y), x - 0.25, x + 0.25, color='tab:blue', lw=2)
    ax.plot(x, LOGMD_OBS[tid], 's', ms=7, color='k', **obs_mstyle(tid))
ax.set(xticks=range(len(_tids)), xticklabels=_tids,
       ylabel=r'$\log\,M_{\rm dust}\,[M_\odot]$',
       title='simulated analogue dust masses (blue) vs ALMA-C11 '
             '(black; hollow = control)')
ax.tick_params(axis='x', rotation=60)
ax.grid(alpha=0.2, axis='y', lw=0.5)
ax.legend(handles=obs_proxy(), frameon=False, fontsize=9, loc='best')
fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, 'dust_mass_almac11.png'), dpi=180)
plt.show()


## Part 6 · does the observed SED need an AGN? does it need to be obscured?

Model ladder per target, all against the CIGALE best-model fluxes: `dust_on` non-AGN hosts,
`dust_on` AGN hosts (**no AGN light** — `BH_SED=False`, so this isolates the host of an AGN),
then `nenkova_i30/i60/i90` = same AGN hosts **plus** the CLUMPY-transmitted AGN
(i=30 ≈ face-on/unobscured, i=90 = edge-on/obscured). Small-multiple SEDs, then band-level
residuals `Δlog F = log10(model stack / obs)` aggregated into rest-frame regimes.
Read-off: if only a Nenkova arm closes the MIR, SIMBA needs the AGN; the inclination that
minimises |Δlog| in the MIR says how obscured it must be. (i30 rows appear once that arm's
RT is run and the source map re-censused.)

In [ ]:
# ── Part 6 · AGN necessity + obscuration, against the observed bands ─────────
LADDER = (('host, non-AGN', 'dust_on', 'no_AGN', 'tab:blue'),
          ('host, AGN (no AGN light)', 'dust_on', 'AGN', 'tab:orange'),
          ('AGN i=30 (unobscured)', 'nenkova_i30', 'AGN', COL['nenkova_i30']),
          ('AGN i=60', 'nenkova_i60', 'AGN', COL['nenkova_i60']),
          ('AGN i=90 (obscured)', 'nenkova_i90', 'AGN', COL['nenkova_i90']))
REGIMES = (('opt <1um', 0.0, 1.0), ('NIR 1-3um', 1.0, 3.0),
           ('MIR 3-30um', 3.0, 30.0), ('FIR >30um', 30.0, np.inf))

_tids6 = [t for t in TARGETS if t in SELECTED and t in OBS]
DLOG = {lab: {rn: [] for rn, _, _ in REGIMES} for lab, _, _, _ in LADDER}

ncol6 = 4
nrow6 = int(np.ceil(len(_tids6) / ncol6))
fig, axes = plt.subplots(nrow6, ncol6, figsize=(4.4 * ncol6, 3.3 * nrow6))
for ax, tid in zip(np.ravel(axes), _tids6):
    A = SELECTED[tid]
    A_cls = _as_str(A['AGN_CLASS'])
    z = TARGETS[tid]['z']
    WOBS, FAC = obs_fnu_factor(z)
    ok = OBS[tid]['fnu'] > 0
    wl_o, f_o = OBS[tid]['wl'][ok], OBS[tid]['fnu'][ok]
    for lab, arm, cls, c_ in LADDER:
        S = stack_curves(A[A_cls == cls], arm)
        if S is None:
            continue
        mod = S['med'] * FAC
        fin = np.isfinite(mod) & (mod > 0)
        ax.plot(WOBS[fin], mod[fin], color=c_, lw=1.3,
                label=f'{lab} (N={S["n"]})' if tid == _tids6[0] else None)
        mod_at = 10 ** np.interp(np.log10(wl_o), np.log10(WOBS[fin]), np.log10(mod[fin]))
        d = np.log10(mod_at / f_o)
        rest = wl_o / (1.0 + z)
        for rn, lo, hi in REGIMES:
            DLOG[lab][rn].extend(d[(rest >= lo) & (rest < hi) & np.isfinite(d)])
    ax.plot(wl_o, f_o, 'o', ms=4, mfc='w', mec='k', zorder=5)
    ax.set(xscale='log', yscale='log', xlim=(0.2, 2000),
           ylim=(f_o.min() / 3e2, f_o.max() * 3e2), title=f'{tid} (z={z:.3f})')
    ax.grid(alpha=0.15, which='both', lw=0.4)
for ax in np.ravel(axes)[len(_tids6):]:
    ax.set_axis_off()
np.ravel(axes)[0].legend(fontsize=7, frameon=False, loc='lower center')
fig.text(0.5, 0.002, r'observed-frame wavelength [$\mu$m]', ha='center')
fig.text(0.002, 0.5, r'$F_\nu$ [mJy]', va='center', rotation='vertical')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'agn_ladder_seds.png'), dpi=160, bbox_inches='tight')
plt.show()

# regime summary across all targets x bands
_lcol = {lab: c_ for lab, _, _, c_ in LADDER}
labs = [lab for lab, _, _, _ in LADDER if any(len(DLOG[lab][rn]) for rn, _, _ in REGIMES)]
xs = np.arange(len(REGIMES))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for k, lab in enumerate(labs):
    off = (k - (len(labs) - 1) / 2) * 0.15
    ab = [np.nanmedian(np.abs(DLOG[lab][rn])) if len(DLOG[lab][rn]) else np.nan
          for rn, _, _ in REGIMES]
    sg = [np.nanmedian(DLOG[lab][rn]) if len(DLOG[lab][rn]) else np.nan for rn, _, _ in REGIMES]
    axes[0].bar(xs + off, ab, 0.15, color=_lcol[lab], label=lab)
    axes[1].bar(xs + off, sg, 0.15, color=_lcol[lab])
axes[1].axhline(0, color='0.4', lw=0.8)
for ax, t_ in zip(axes, (r'median $|\Delta\log F|$ (accuracy)',
                         r'median $\Delta\log F$ (bias: model $-$ obs)')):
    ax.set(xticks=xs, xticklabels=[rn for rn, _, _ in REGIMES], title=t_)
    ax.grid(alpha=0.2, axis='y', lw=0.5)
axes[0].set_ylabel('dex')
axes[0].legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'agn_ladder_residuals.png'), dpi=180, bbox_inches='tight')
plt.show()

print('median dlog10(model/obs) per regime  [negative = model too faint]')
print(f'{"model":26s} ' + ' '.join(f'{rn:>12s}' for rn, _, _ in REGIMES))
for lab in labs:
    v = ' '.join(f'{(np.nanmedian(DLOG[lab][rn]) if len(DLOG[lab][rn]) else np.nan):12.2f}'
                 for rn, _, _ in REGIMES)
    print(f'{lab:26s} {v}')

## Part 7 · if the dust ratios match, does the attenuation match?

The analogues were *selected* in the age–mass windows where SIMBA predicts dusty sources
(dust-to-molecular ratio > 10⁻³, controls < 10⁻⁴). Here: per-galaxy `A_V` (from Part 3's
dust_on/dust_off difference, `findx0`) against `log M_dust/M*` and `log M_dust/M_gas`,
with the observed sources overplotted (`Av_ISM` vs the sanitized dust masses of Part 5,
and vs the CSV `logDGR`). Caveat: the selection tables carry **total** gas, not H₂, so the
sim x-axis of the right panel is a lower bound on the dust-to-molecular ratio; the dashed
and dotted guides are the 10⁻³ / 10⁻⁴ selection thresholds. Depends on Part 5 (`LOGMD_OBS`). Part 9d redoes the figure like-for-like with the mock CIGALE `Av_ISM` (needs the Part 9 fits).


In [ ]:
# ── Part 7 · dust ratios vs attenuation ──────────────────────────────────────
try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

AV_ALL = {}                                # (snap,gid) -> A_V at findx0, union-wide
for r in UNION:
    k = (int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT']))
    c_ = attenuation_curve(*k)
    if c_ is not None and np.isfinite(c_[_jV]):
        AV_ALL[k] = float(c_[_jV])
print(f'A_V available for {len(AV_ALL)}/{len(UNION)} union galaxies (needs dust_on+dust_off)')

_rows7 = {}
for tid in SELECTED:
    for r in SELECTED[tid]:
        _rows7.setdefault((int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])), r)
ds, dg, av, isagn = [], [], [], []
for k, r in _rows7.items():
    if k not in AV_ALL:
        continue
    md, mg = float(r['MDUST']), float(r['MGAS'])
    ds.append(np.log10(md) - float(r['LOG_MSTAR']) if md > 0 else np.nan)
    dg.append(np.log10(md / mg) if (md > 0 and mg > 0) else np.nan)
    av.append(AV_ALL[k])
    isagn.append(str(r['AGN_CLASS']) == 'AGN')
ds, dg, av, isagn = map(np.asarray, (ds, dg, av, isagn))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)
for flag, c_, lab in ((False, 'tab:blue', 'non-AGN host'), (True, 'tab:orange', 'AGN host')):
    m = isagn == flag
    axes[0].plot(ds[m], av[m], 'o', ms=4, color=c_, alpha=0.55, mec='none', label=lab)
    axes[1].plot(dg[m], av[m], 'o', ms=4, color=c_, alpha=0.55, mec='none')
for tid, T in TARGETS.items():             # observed side
    if T['row'] is None or not np.isfinite(T['av_ism']):
        continue
    if tid in LOGMD_OBS and np.isfinite(LOGMD_OBS[tid]):
        axes[0].errorbar(LOGMD_OBS[tid] - T['logm'], T['av_ism'], yerr=T['av_ism_err'],
                         fmt='s', ms=6, color='k', capsize=2, zorder=5,
                         **obs_mstyle(tid))
    dgr = float(T['row'].get('logDGR', np.nan))
    if np.isfinite(dgr):
        axes[1].errorbar(dgr, T['av_ism'], yerr=T['av_ism_err'], fmt='s', ms=6,
                         color='k', capsize=2, zorder=5, **obs_mstyle(tid))
axes[1].axvline(-3, color='0.6', ls='--', lw=0.8)      # dusty-selection threshold
axes[1].axvline(-4, color='0.6', ls=':', lw=0.8)       # control threshold
axes[0].set_xlim(-5)
axes[1].set_xlim(-4)
axes[0].set(xlabel=r'$\log\,M_{\rm dust}/M_\star$',
            ylabel=r'$A_V$ [mag] (sim)  /  Av_ISM (obs, black)')
axes[1].set(xlabel=r'sim: $\log\,M_{\rm dust}/M_{\rm gas,tot}$   ·   '
                   r'obs: logDGR ($M_{\rm dust}/M_{\rm mol}$)')
for ax in axes:
    ax.grid(alpha=0.2, lw=0.5)
axes[0].legend(handles=[Line2D([], [], ls='none', marker='o', ms=4, color=c_,
                               alpha=0.55, mec='none', label=l_)
                        for c_, l_ in (('tab:blue', 'non-AGN host'),
                                       ('tab:orange', 'AGN host'))] + obs_proxy(),
               frameon=False, fontsize=9)
fig.suptitle('does matching the dust ratios reproduce the attenuation?')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'dust_ratio_vs_av.png'), dpi=180, bbox_inches='tight')
plt.show()

for name, x in (('log Md/M*', ds), ('log Md/Mgas', dg)):
    m = np.isfinite(x) & np.isfinite(av)
    if spearmanr is not None and m.sum() > 3:
        rho, p = spearmanr(x[m], av[m])
        print(f'Spearman A_V vs {name}: rho={rho:+.2f} (p={p:.1e}, N={m.sum()})')

## Part 8 · host viewing angle — does attenuation need a particular inclination?

The dust arms carry 4 quasi-orthogonal sightlines (`THETA=[0,45,90,135]`). Per galaxy:
`A_V` per sightline (matched dust_on/dust_off index). Left: sightline-to-sightline spread vs
the median. Right, per target: the gap `Av_ISM(obs) − median sim A_V` (grey bars) against the
typical ± half sightline range of its members (red) — if the red bars are much shorter than
the grey ones, no viewing angle reconciles sim and observed attenuation, and the dust
content/geometry itself must differ. (Torus inclination is Part 6's axis, not this one.)
Depends on Part 7 (`AV_ALL`).
A third figure plots each member's $\log M_{\rm dust}$ against its $A_V$ **split by the
four sightlines** (colored by $\theta$): does more dust mean more attenuation regardless
of viewing angle, or does geometry scatter the relation? `AV_SIGHT` keeps the
sightline-indexed vector (NaN where a sightline's flux ratio is unusable), so the colors
identify the same $\theta$ across galaxies. Also depends on Part 7's `spearmanr`.


In [ ]:
# ── Part 8 · A_V across the 4 dust-arm sightlines ────────────────────────────
def _sed_allsight(arm, snap, gid):
    p = rtout_path(arm, snap, gid)
    if p is None:
        return None
    sed = ModelOutput(p).get_sed(inclination='all', aperture=-1)
    wav = np.asarray(sed.wav, float)
    val = np.atleast_2d(np.asarray(sed.val, float))
    s_ = np.argsort(wav)
    return wav[s_], val[:, s_]

AV_SIGHT = {}                              # (snap,gid) -> A_V per sightline
for k in AV_ALL:
    on, off = _sed_allsight('dust_on', *k), _sed_allsight('dust_off', *k)
    if on is None or off is None:
        continue
    n = min(on[1].shape[0], off[1].shape[0])
    with np.errstate(invalid='ignore', divide='ignore'):
        v = np.array([(-2.5 * np.log10(on_grid(on[0], on[1][s_]) /
                                       on_grid(off[0], off[1][s_])))[_jV] for s_ in range(n)])
    if np.isfinite(v).sum() >= 2:
        AV_SIGHT[k] = v                    # sightline-indexed, may hold NaN

med8 = np.array([np.nanmedian(v) for v in AV_SIGHT.values()])
rng8 = np.array([np.nanmax(v) - np.nanmin(v) for v in AV_SIGHT.values()])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
axes[0].plot(med8, rng8, 'o', ms=4, color='tab:blue', alpha=0.6, mec='none')
axes[0].set(xlabel=r'median $A_V$ across sightlines [mag]',
            ylabel=r'max$-$min $A_V$ [mag]',
            title=f'sightline spread per galaxy (N={len(AV_SIGHT)})')

_t8 = [t for t in TARGETS if TARGETS[t]['sample'] == 'almac11'
       and np.isfinite(TARGETS[t]['av_ism']) and t in SELECTED]
gaps, sprd = [], []
for tid in _t8:
    ks = [(int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])) for r in SELECTED[tid]]
    vs = [AV_SIGHT[k] for k in ks if k in AV_SIGHT]
    if not vs:
        gaps.append(np.nan)
        sprd.append(np.nan)
        continue
    gaps.append(TARGETS[tid]['av_ism'] - np.median([np.nanmedian(v) for v in vs]))
    sprd.append(np.median([(np.nanmax(v) - np.nanmin(v)) / 2 for v in vs]))
gaps, sprd = np.asarray(gaps), np.asarray(sprd)
x8 = np.arange(len(_t8))
axes[1].bar(x8, gaps, 0.6, color='0.75', label=r'Av_ISM(obs) $-$ median sim $A_V$')
axes[1].errorbar(x8, np.zeros(len(_t8)), yerr=sprd, fmt='none', ecolor='tab:red',
                 capsize=2, label=r'$\pm$ half sightline range')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set(xticks=x8, xticklabels=_t8, ylabel='[mag]',
            title='can viewing angle bridge the gap?')
axes[1].tick_params(axis='x', rotation=60)
axes[1].legend(fontsize=8, frameon=False)
for ax in axes:
    ax.grid(alpha=0.2, lw=0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'sightline_av_spread.png'), dpi=180, bbox_inches='tight')
plt.show()
print(f'median sightline A_V spread (max-min): {np.median(rng8):.3f} mag | '
      f'median |obs - sim| A_V gap: {np.nanmedian(np.abs(gaps)):.2f} mag')

# ── dust mass vs attenuation, split by the 4 sightlines ──────────────────────
THETA8 = (0, 45, 90, 135)
_rows8 = {}                                # (snap,gid) -> selection row, deduped
for tid in SELECTED:
    for r in SELECTED[tid]:
        _rows8.setdefault((int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])), r)
fig, ax = plt.subplots(figsize=(7, 5.2))
for s_ in range(4):
    x_, y_ = [], []
    for k, r in _rows8.items():
        v = AV_SIGHT.get(k)
        if v is None or s_ >= v.size or not np.isfinite(v[s_]):
            continue
        md = float(r['MDUST'])
        if md > 0:
            x_.append(np.log10(md))
            y_.append(float(v[s_]))
    x_, y_ = np.asarray(x_), np.asarray(y_)
    ax.plot(x_, y_, 'o', ms=4, color=f'C{s_}', alpha=0.55, mec='none',
            label=fr'$\theta={THETA8[s_]}^\circ$')
    if spearmanr is not None and x_.size > 3:
        rho, p = spearmanr(x_, y_)
        print(f'theta={THETA8[s_]:3d}: Spearman A_V vs log Mdust '
              f'rho={rho:+.2f} (p={p:.1e}, N={x_.size})')
ax.set(xlabel=r'$\log\,M_{\rm dust}$ [$M_\odot$]', ylabel=r'$A_V$ [mag]',
       title='dust mass vs attenuation per sightline')
ax.grid(alpha=0.2, lw=0.5)
ax.legend(fontsize=9, frameon=False, title='sightline')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'dustmass_vs_av_sightlines.png'), dpi=180)
plt.show()


## Part 9 · consistent A_V — CIGALE fits of the mock dust_on photometry

Part 3 compares two **different quantities**: the RT differential
$A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ (emergent, *total* effective attenuation) against
the observed `bayes.attenuation.Av_ISM` — a **parameter** of the modified Charlot & Fall
model, and only its **ISM component** (with the CF00 default $\mu=0.44$ the young populations
see $A_V^{\rm ISM}/\mu \approx 2.3\times$ more). Part of the Part 3 discrepancy is therefore
definitional. This part closes the loop by pushing the mocks through **the same estimator as
the observations**:

- **9a** — observed-frame photometry of every union galaxy's `dust_on` SED (`findx0`,
  total aperture) in **exactly the 19 bands** of the observed CIGALE fit (MegaCam u/g,
  Suprime B/V/r/i + IB427/464, VISTA JHKs, IRAC 1–4, MIPS 24, PACS 100, SPIRE 250,
  ALMA band 6), with the Hyperion MC errors propagated, plus each SED's rest-frame
  8–1000 µm dust luminosity (the 9b ⟨U⟩ closure diagnostic uses it). Source-map driven
  (the dust_on arm spans the new `sed_almac11` tree **and** two reused CEERS trees),
  reusing the `MakeSED.extract_flux_batch` primitives.
- **9a2** — the **ALMA gate** (only sources with mock band-6 flux $> 5\,\mu$Jy are fitted —
  the observed sample is ALMA-selected, and below that the FIR side of the energy balance
  is unconstrained) and the **smoothed SFH archive**: every union galaxy's archaeological
  SFH (star-particle formation times from its RT cutout), smoothed →
  `cigale/almac11_sfh_smoothed.h5`, which 9b injects into CIGALE directly. Each SFH is
  also still fitted with the `sfhdelayedbq` form (m25 7d machinery) as a descriptive
  record of the family's `tau_main`/`age_main`/`age_bq`/`r_sfr`.
- **9b** — CIGALE 2025 inputs + run dirs (**one per snapshot × bc03 metallicity node**) +
  **one SLURM job array**. Module chain: `sfhfromfile` + `bc03` (Chabrier) + `nebular` +
  `dustatt_modified_CF00` + `dl2014` + rest-frame UVJ. Priors (2026-08-06 redesign —
  replaces the sfhdelayedbq quantile grids of the `_dw` runs): the **SFH is injected**,
  not parametrized — each run's `sfh.fits` carries its members' smoothed archaeological
  SFHs (1 Myr grid) and CIGALE marginalizes over that family, shape-only
  (`normalise=True`) so $M_*$ stays the fitted normalization and an honest 9c recovery
  check; the **stellar metallicity** is pinned to the nearest allowed bc03 node
  (`cg.split_by_metallicity`, gas Z → ≤3 `zgas` nodes); the **dust-emission radiation
  field is free** (2026-08-06b, replacing the semi-pinned ⟨U⟩ scheme — `umin` on an
  11-node ladder, `gamma` 4 nodes, `qpah` 2 nodes; the SIMBA $\langle U\rangle =
  L_{\rm dust}/(125\,M_{\rm dust})$ is a 9c closure diagnostic, not a grid pin — dust
  emission never touches the UV/optical, so the freedom reopens no age–dust degeneracy,
  it only stops the IR bands torquing $A_V$ through energy balance); the **attenuation
  stays the free measurand** on a denser 16-node `Av_ISM` grid (µ and `slope_BC` at the
  CF00 defaults, `slope_ISM` freed to greyer options — `Av_ISM` is normalized at V, so
  `bayes.attenuation.Av_ISM` keeps the observed column's meaning).
- **9c** — after the array drains: (i) the Part 3 figure **like-for-like** — mock
  `bayes.attenuation.Av_ISM` vs observed `Av_ISM`; (ii) the **estimator-closure** test only a
  simulation can do — CIGALE's total V attenuation `attenuation.generic.bessell.V` (the
  stock-DB name of the observed `V_B90` column) and `Av_ISM` against the RT truth per galaxy,
  i.e. the bias of the estimator itself; (iii) **truth recovery** — recovered $\log M_*$ and
  mass-weighted age vs caesar, the posterior `umin`/`dust.umean` against the SIMBA
  $\langle U\rangle$ (radiation-field closure: recovering ⟨U⟩ validates the retired
  anchored grids; landing colder is the $L_{\rm dust}$ deficit story), plus the `Av_ISM`
  shift against the archived prior-free runs.

**Run order.** 9a (reads ~480 rtouts, ≈10–15 min) → 9a2 (reads the particle cutouts, a few
minutes) → 9b, all in the cluster kernel → `sbatch` the printed job (one task per
snapshot × Z group, minutes each) → 9c (needs Part 3 in-session for `attenuation_curve`).

**Caveats.** The priors deliberately break strict estimator symmetry with the observed fit
(which had no SFH/Z truth): the prior-informed fit measures the attenuation bias with the
stellar population constrained to SIMBA's own SFH family, isolating the dust side of the
degeneracy. The archaeological SFHs weight by *current* particle masses (mass return tilts
old bins low — a shape effect only, never used as a mass prior). Injecting them via
`sfhfromfile` removes the parametric-form mismatch the `_dw` runs had (`delayedbq` cannot
follow SIMBA's bursty tracks in detail — the per-galaxy `r2` in
`cigale/almac11_sfh_delayedbq_fits.fits` quantifies that). The original prior-free runs
(`dust_on_findx0_snapNNN`, no `_Zs` suffix) stay on disk as the baseline — 9c auto-detects
the Z-tagged runs and prints the shift between the two; they were fitted on the **ungated**
sample, so the shift is computed on the gated intersection. Single sightline `findx0`
(Part 3 convention — Part 8 shows the sightline spread is small); NaN MC errors are
repaired to 10% of the flux and CIGALE adds `additionalerror = 0.1` in quadrature at fit
time.

In [ ]:
# ── Part 9a · mock photometry: observed-frame fluxes in the 19 ALMA-C11 bands ─
# Source-map driven: the dust_on arm spans 3 RT trees (sed_almac11 + 2 reused
# CEERS trees), so MakeSED.extract_flux_batch's fixed snap_XXX/gal_X path
# convention cannot be pointed at one tree — this loop reuses its exact
# primitives instead (same unit conversion + MC-error propagation).
# Needs only Part 0.
# 2026-08-06c: emits TWO flux tables per galaxy — the observational-like
# findx0 sightline AND the 4-sightline mean ('smean'). CIGALE's energy
# balance is isotropic while powderday's is not: averaging the sightlines
# isotropizes the mock, so 9b/9e can separate sightline anisotropy from
# model rigidity in the chi2 tail.
from simbanator.sed.makesed import _read_sed, _sed_to_mJy
from simbanator.sed.flux_extraction import (get_svo_filters, load_local_filters,
                                            flux_extraction)

CIGDIR = os.path.join(OUTDIR, 'cigale')
os.makedirs(CIGDIR, exist_ok=True)
SIGHTLINE_FINDX = 0            # findx0 — same convention as the Part 3 A_V
SL_TAGS9 = (f'findx{SIGHTLINE_FINDX}', 'smean')

# the 19 bands of the observed CIGALE fit; SVO fetch per instrument. The
# Suprime narrowbands are deliberately NOT fetched: at 2500 SED points the
# IB427/IB464 filters get ~8 native points across (enough for the trapezoid
# convolution) but the ~7 nm NB ones would be under-sampled.
SVO_SETS = [
    ('Subaru',   'Suprime', ['B', 'V', 'r', 'i', 'IB427', 'IB464']),
    ('CFHT',     'MegaCam', ['u', 'g']),
    ('Paranal',  'VIRCAM',  ['J', 'H', 'Ks']),
    ('Spitzer',  'IRAC',    ['I1', 'I2', 'I3', 'I4']),
    ('Spitzer',  'MIPS',    ['24mu']),
    ('Herschel', 'PACS',    ['green']),
    ('Herschel', 'SPIRE',   ['PSW']),
]
LOCAL_FILTERS = {   # custom top-hat; its CIGALE twin 'alma.band6' is registered
    'ALMA': {'ALMA': {'band6': os.path.join(HOME, 'ALMA_band6.res')}},
}
profiles = {}
for _fac, _inst, _filts in SVO_SETS:
    for f_, d_ in get_svo_filters(_fac, _inst, filters=_filts,
                                  wave_unit='micron').items():
        for i_, fd_ in d_.items():
            profiles.setdefault(f_, {}).setdefault(i_, {}).update(fd_)
for f_, d_ in load_local_filters(LOCAL_FILTERS, 'micron').items():
    for i_, fd_ in d_.items():
        profiles.setdefault(f_, {}).setdefault(i_, {}).update(fd_)
_nfilt = sum(len(fd_) for d_ in profiles.values() for fd_ in d_.values())
print(f'{_nfilt} filter profiles ready (expect 19)')

# snap -> z (identical for every member of a snapshot)
ZSNAP = {int(r['SNAPSHOT']): float(r['REDSHIFT'])
         for t_ in SELECTED.values() for r in t_}

rows9 = {tag_: [] for tag_ in SL_TAGS9}
skip9, fail9 = [], []
for kk, r in enumerate(UNION):
    snap, gid = int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])
    p = rtout_path('dust_on', snap, gid)
    if p is None:
        skip9.append((snap, gid)); continue
    try:
        wav_raw, flux_raw, unc_raw = _read_sed(p, aperture=-1,
                                               uncertainties=True)
        z = ZSNAP[snap]
        wav, flux = _sed_to_mJy(wav_raw, flux_raw, z, apply_redshift=True)
        # flux/unc stay the _sed_to_mJy astropy Quantities — flux_extraction
        # needs the units (np.asarray would strip them and every filter would
        # be silently skipped, leaving a 4-column flux table)
        if flux.ndim == 2:                              # (n_sight, n_wave)
            fx_v = {SL_TAGS9[0]: flux[SIGHTLINE_FINDX],
                    'smean': flux.mean(axis=0)}
        else:
            fx_v = {SL_TAGS9[0]: flux, 'smean': flux}
        ux_v = {tag_: None for tag_ in SL_TAGS9}
        if unc_raw is not None:
            _, unc = _sed_to_mJy(wav_raw, unc_raw, z, apply_redshift=True)
            if unc.ndim == 2:
                ux_v[SL_TAGS9[0]] = unc[SIGHTLINE_FINDX]
                # MC errors independent per sightline -> error of the mean
                ux_v['smean'] = np.sqrt((unc ** 2).sum(axis=0)) / unc.shape[0]
            else:
                ux_v[SL_TAGS9[0]] = ux_v['smean'] = unc
        # rest-frame 8-1000 um dust luminosity: integral of nuLnu dln(lambda)
        # on the raw powderday arrays (nuLnu [erg/s] vs rest um) — the 9b
        # <U> = Ldust / (125 Mdust) closure diagnostic uses it
        fr2 = np.atleast_2d(np.asarray(flux_raw, float))
        raw_v = {SL_TAGS9[0]: fr2[SIGHTLINE_FINDX] if fr2.shape[0] > 1
                 else fr2[0],
                 'smean': fr2.mean(axis=0)}
        wraw = np.asarray(wav_raw, float)
        _mir = (wraw >= 8.0) & (wraw <= 1000.0)
        _o = np.argsort(wraw[_mir])
        for tag_ in SL_TAGS9:
            res9 = flux_extraction(None, None, wav, fx_v[tag_],
                                   wave_unit='micron', filter_list=profiles,
                                   flux_unc=ux_v[tag_])
            ldust = float(np.trapz(raw_v[tag_][_mir][_o],
                                   np.log(wraw[_mir][_o])))
            row = {'gal_id_at_snap': gid, 'snap': snap, 'redshift': z,
                   'Ldust_8_1000_Lsun': ldust / 3.828e33}
            for fac_, inst_d in res9.items():
                for inst_, filt_d in inst_d.items():
                    for fn_, fd_ in filt_d.items():
                        col = f'{fac_}.{inst_}.{fn_}'
                        row[col] = fd_['mJy']
                        row[f'{col}_err'] = fd_.get('mJy_err', np.nan)
            rows9[tag_].append(row)
    except Exception as e:
        fail9.append((snap, gid, str(e))); continue
    if (kk + 1) % 50 == 0:
        print(f'  {kk + 1}/{len(UNION)} rtouts read')

for tag_ in SL_TAGS9:
    ftag = os.path.join(CIGDIR, f'almac11_fluxes_dust_on_{tag_}.fits')
    Table(rows9[tag_]).write(ftag, overwrite=True)
    nb9 = (len(rows9[tag_][0]) - 4) // 2 if rows9[tag_] else 0
    print(f'{len(rows9[tag_])} galaxies x {nb9} bands -> {ftag}')
FLUX_FITS9 = os.path.join(CIGDIR,
                          f'almac11_fluxes_dust_on_findx{SIGHTLINE_FINDX}.fits')
print(f'skipped (RT incomplete): {len(skip9)} | read failures: {len(fail9)}')
for s_ in fail9[:10]:
    print('   fail:', s_)

### Part 9a2 — ALMA gate + SIMBA SFHs → smoothed archive (+ delayed+bq record)

Two inputs 9b needs beyond the photometry:

- **ALMA gate.** The observed sample is ALMA-selected: mocks with band-6 flux
  $\le 5\,\mu$Jy have an unconstrained FIR side and only dilute the comparison. This cell
  reports the per-snapshot counts; the gate itself is applied in 9b (`ALMA_MIN_UJY`), so
  regating never stales the tables built here.
- **Smoothed SFH archive.** Each galaxy's archaeological SFH — mass formed per 100 Myr
  from the star-particle formation times in its RT cutout (`filtered_particles`, the
  exact particles the RT saw) — is smoothed (`smooth_resample_sfh`, 25 Myr grid, 150 Myr
  kernel) and stored in `cigale/almac11_sfh_smoothed.h5` (one dataset per id:
  `[t_gyr, SFR]` + `t_obs_gyr`). 9b **injects these SFHs directly into CIGALE** via the
  `sfhfromfile` module — no parametric SFH approximation in the fit anymore. Each SFH is
  also still fitted with the `sfhdelayedbq` form (`fit_delayed_bq` →
  `cigale/almac11_sfh_delayedbq_fits.fits`), now purely a descriptive record of the
  family's shape parameters (and the reference the archived `_dw` parametric runs used).

Caveat: the archaeological SFH weights by *current* particle masses, so old bins sit low by
the return fraction (≲30%) — a mild shape tilt, far below the burst noise the smoothing
removes; with `normalise=True` in `sfhfromfile` only the shape enters the fit — no mass or
normalization is ever taken from these SFHs.

In [ ]:
# ── Part 9a2 · ALMA gate + SIMBA SFHs -> smoothed archive (+bq record) ───────
# Self-contained after Part 0 + the Part 9a FITS. For every union galaxy the
# archaeological SFH (mass formed per 100 Myr, from the star-particle
# formation times in its RT cutout) is smoothed and (a) PERSISTED to SFH_H5 —
# 9b injects these directly into CIGALE via sfhfromfile — and (b) fitted with
# the delayed+bq form (m25 Part 7d machinery), now a descriptive record only.
# ALL galaxies are processed — the ALMA gate is applied in 9b, so regating
# never stales the archive.
from simbanator.analysis.sfh_utils import (smooth_resample_sfh,
                                           sfr_delayed_bq, fit_delayed_bq)

CIGDIR       = os.path.join(OUTDIR, 'cigale')
FLUX_FITS9   = os.path.join(CIGDIR, 'almac11_fluxes_dust_on_findx0.fits')
SFH_FITS9    = os.path.join(CIGDIR, 'almac11_sfh_delayedbq_fits.fits')
SFH_H5       = os.path.join(CIGDIR, 'almac11_sfh_smoothed.h5')
PARTDIR      = os.path.join(HOME, 'output', 'cis100', 'filtered_particles')
ALMA_MIN_UJY = 5.0        # 9b fits only sources above this
SFH_BIN_MYR  = 100.0      # archaeological SFH bin width

_t9a  = Table.read(FLUX_FITS9)
_alma = 1e3 * np.asarray(_t9a['ALMA.ALMA.band6'], float)      # mJy -> uJy
_gate = np.isfinite(_alma) & (_alma > ALMA_MIN_UJY)
print(f'[gate] ALMA band 6 > {ALMA_MIN_UJY:g} uJy: {int(_gate.sum())}/'
      f'{len(_t9a)} pass (median {np.nanmedian(_alma):.2f} uJy)')
for s_ in sorted(set(np.asarray(_t9a['snap'], int).tolist())):
    m_ = np.asarray(_t9a['snap'], int) == s_
    print(f'   snap {s_:03d}: {int((_gate & m_).sum()):3d}/{int(m_.sum()):3d}')

# scale factor -> cosmic time on one interpolation grid (per-particle
# Planck13.age() calls would take minutes)
_agrid = np.linspace(0.02, 1.0, 600)
_tgrid = Planck13.age(1.0 / _agrid - 1.0).to(u.Gyr).value

def _archaeo_sfh(snap, gid, t_obs_gyr):
    # (t_gyr, sfr) from the cutout's star particles, or None. Current masses
    # -> old bins low by the return fraction (shape prior only).
    p = os.path.join(PARTDIR, f'snap_{int(snap):03d}',
                     f'm100n1024_snap{int(snap):03d}_gal{int(gid):06d}.h5')
    if not os.path.exists(p):
        return None
    with h5py.File(p, 'r') as f:
        if 'PartType4' not in f:
            return None
        a_ = np.asarray(f['PartType4/StellarFormationTime'][:], float)
        m_ = np.asarray(f['PartType4/Masses'][:], float)
        h_ = float(f['Header'].attrs['HubbleParam'])
    ok = np.isfinite(a_) & (a_ > 0) & (a_ <= 1)
    if ok.sum() < 20:
        return None
    tf = np.interp(a_[ok], _agrid, _tgrid)                    # Gyr
    mm = m_[ok] * 1e10 / h_                                   # Msun
    dt = SFH_BIN_MYR / 1e3
    edges = np.arange(0.0, t_obs_gyr + dt, dt)
    hist, _ = np.histogram(tf, bins=edges, weights=mm)
    return 0.5 * (edges[:-1] + edges[1:]), hist / (dt * 1e9)  # Msun/yr

TOBS9 = {s_: Planck13.age(z_).to(u.Gyr).value for s_, z_ in
         {int(r['snap']): float(r['redshift']) for r in _t9a}.items()}
rows9s, _ex9, _sfh_arch, _nofit = [], [], {}, 0
for kk, r in enumerate(_t9a):
    snap, gid = int(r['snap']), int(r['gal_id_at_snap'])
    t_obs = TOBS9[snap]
    sfh = _archaeo_sfh(snap, gid, t_obs)
    bq = None
    if sfh is not None:
        ts, ss = smooth_resample_sfh(*sfh, dt_myr=25.0, kernel_myr=150.0)
        _sfh_arch[f'snap{snap:03d}_gal{gid}'] = (ts, ss, t_obs)
        bq = fit_delayed_bq(ts, ss, t_obs)
    if bq is None:
        _nofit += 1
    elif _gate[kk] and len(_ex9) < 6:
        _ex9.append((f'snap{snap:03d}_gal{gid}', sfh, ts, ss, bq, t_obs))
    rows9s.append(dict(
        id=f'snap{snap:03d}_gal{gid}', snap=snap, gal_id=gid,
        z=float(r['redshift']), alma_uJy=float(_alma[kk]),
        tau_main_myr=bq['tau_main_myr'] if bq else np.nan,
        age_main_myr=bq['age_main_myr'] if bq else np.nan,
        age_bq_myr=bq['age_bq_myr'] if bq else np.nan,
        r_sfr=bq['r_sfr'] if bq else np.nan,
        r2=bq['r2'] if bq else np.nan))
    if (kk + 1) % 100 == 0:
        print(f'  {kk + 1}/{len(_t9a)} SFHs fitted')
T9S = Table(rows9s)
T9S.write(SFH_FITS9, overwrite=True)

# smoothed-SFH archive for 9b's sfhfromfile injection: one dataset per id,
# columns [t_gyr (cosmic time), SFR (Msun/yr)] on the 25 Myr smoothed grid
with h5py.File(SFH_H5, 'w') as f5:
    f5.attrs['dt_myr'] = 25.0
    f5.attrs['kernel_myr'] = 150.0
    for id_, (ts_, ss_, tob_) in _sfh_arch.items():
        d5 = f5.create_dataset(id_, data=np.column_stack([ts_, ss_]))
        d5.attrs['t_obs_gyr'] = tob_
print(f'[SFH archive] {len(_sfh_arch)}/{len(T9S)} smoothed SFHs -> {SFH_H5}')
print(f'[SFH record] delayed+bq ok for {len(T9S) - _nofit}/{len(T9S)} '
      f'galaxies -> {SFH_FITS9}')
print('   parameter          p5       p50       p95   (gated members only)')
for c_ in ('tau_main_myr', 'age_main_myr', 'age_bq_myr', 'r_sfr', 'r2'):
    v_ = np.asarray(T9S[c_], float)[_gate]
    v_ = v_[np.isfinite(v_)]
    a_, b_, c2_ = (np.percentile(v_, [5, 50, 95]) if v_.size
                   else (np.nan,) * 3)
    print(f'   {c_:<14s} {a_:9.2f} {b_:9.2f} {c2_:9.2f}')

# ── figures: example fits (gated) + fitted-parameter distributions ──
if _ex9:
    fig, axs = plt.subplots(2, 3, figsize=(13.5, 6.6), squeeze=False)
    for k_, (id_, sfh_, ts_, ss_, bq_, tob_) in enumerate(_ex9):
        ax = axs[k_ // 3][k_ % 3]
        ax.plot(sfh_[0], sfh_[1], 'o', ms=2.2, color='0.7',
                label='archaeological (100 Myr)')
        ax.plot(ts_, ss_, '-', color='0.25', lw=1.3, label='smoothed')
        tt_ = np.linspace(ts_[0], ts_[-1], 300)
        ax.plot(tt_, sfr_delayed_bq(tt_, bq_['A'], bq_['tau_main_myr'] / 1e3,
                bq_['age_main_myr'] / 1e3, bq_['age_bq_myr'] / 1e3,
                bq_['r_sfr'], tob_), '-', color='#D55E00', lw=1.7,
                label='delayed+bq')
        ax.set_title(f'{id_}  ' + r'$\tau$=%.0f age=%.0f bq=%.0f r=%.2f '
                     r'$R^2$=%.2f' % (bq_['tau_main_myr'],
                     bq_['age_main_myr'], bq_['age_bq_myr'], bq_['r_sfr'],
                     bq_['r2']), fontsize=8)
        ax.set_xlabel('cosmic time [Gyr]')
        ax.set_ylabel(r'SFR [$M_\odot$/yr]')
        if k_ == 0:
            ax.legend(fontsize=7, frameon=False)
    for k_ in range(len(_ex9), 6):
        axs[k_ // 3][k_ % 3].set_axis_off()
    fig.suptitle('archaeological SFHs with the delayed+bq fit '
                 '(first 6 gated)', y=1.0)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, 'sfh_prior_examples.png'), dpi=150,
                bbox_inches='tight')

fig, axs = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, c_, lab in zip(axs, ('tau_main_myr', 'age_main_myr', 'age_bq_myr',
                             'r_sfr'),
                       ('tau_main [Myr]', 'age_main [Myr]', 'age_bq [Myr]',
                        'r_sfr')):
    v_all = np.asarray(T9S[c_], float)
    for m_, col_, l_ in ((np.isfinite(v_all), '0.75', 'all union'),
                         (_gate & np.isfinite(v_all), '#0072B2',
                          f'ALMA > {ALMA_MIN_UJY:g} uJy')):
        if m_.any():
            ax.hist(v_all[m_], bins=20, color=col_, alpha=0.75,
                    label=l_, histtype='stepfilled')
    ax.set_xlabel(lab)
    ax.set_ylabel('galaxies')
axs[0].legend(fontsize=7, frameon=False)
fig.suptitle('delayed+bq fitted parameters (descriptive record — 9b injects '
             'the smoothed SFHs directly)', y=1.03)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'sfh_prior_distributions.png'), dpi=150,
            bbox_inches='tight')
plt.show()

### Part 9b — CIGALE run dirs (snapshot × Z-group) + ONE SLURM job array

CIGALE never fits inside this kernel: the cell only writes the data files, every
`pcigale.ini` (+`.spec`, via `simbanator.sed.cigale.prepare_run` — no `pcigale
init`/`genconf` by hand) and one sbatch array with **one task per (snapshot, metallicity
group) run**. Grid design (2026-08-06 — the SFH is now *injected*, replacing the
sfhdelayedbq quantile grids of the `_dw` runs):

- **ALMA gate** — only sources with mock band-6 flux > `ALMA_MIN_UJY` (5 µJy) enter the
  inputs; the observed sample is ALMA-selected and fainter mocks leave the FIR side of
  the energy balance unconstrained.
- **`sfhfromfile` with the injected SIMBA SFH family** (9a2 archive): each run writes an
  `sfh.fits` (col 0 = time in Myr, 0…t_universe(z) in strict 1 Myr steps; one column per
  member = its smoothed archaeological SFH interpolated from the 25 Myr archive grid,
  SFR = 0 before the first archived sample, edge-held after the last) and CIGALE
  marginalizes over the member columns (`sfr_column` = all, single `age` per run). With
  `normalise=True` only the SFH *shape* is injected — $M_*$ stays the fitted
  normalization (9c recovery check) and no parametric SFH form sits between SIMBA and
  CIGALE anymore. `best.sfh.index` records which member's SFH won. Members without an
  archived SFH are dropped from the family (their photometry is still fitted against the
  other members' shapes).
- **metallicity pinned**: caesar `metallicities.stellar` snapped to the nearest allowed
  bc03 node, one run per node (`cg.split_by_metallicity`); the SFR-weighted gas Z
  restricts the `nebular` `zgas` grid to ≤3 nodes.
- **radiation field freed** (2026-08-06b — replaces the semi-pinned ⟨U⟩ scheme. The
  fully pinned grids had piled `umin` at the low edge and missed the 4 dust bands at
  3–6σ — MIPS 24 overshot ~30%, PACS/SPIRE/ALMA undershot ×1.5–3 — and even the
  anchored grid ties the FIR shape to the $\langle U\rangle =
  L_{\rm dust}/(125\,M_{\rm dust})$ mapping, so with the SFH fixed any residual FIR
  mismatch leaks into `Av_ISM` through the energy-balance term — the one bias pathway
  the fixed-SFH design cannot afford): `umin` on a free 11-node ladder 0.1–25 snapped
  to the dl2014 nodes, `gamma` ∈ {0.01, 0.02, 0.05, 0.1} (the MIPS 24 overshoot lives
  in the warm/PDR component, which `umin` alone cannot fix), `alpha` = 2 (default),
  `qpah` ∈ {2.50, 5.95} (powderday's `PAH_frac['usg']` = 5.86% + one lower node for
  the rest ~17 µm excess). Dust emission never touches the UV/optical, so the extra
  freedom reopens no age–dust degeneracy — it trades a little IR leverage on `Av_ISM`
  (variance) against the pin's bias. Per-galaxy ⟨U⟩ (Draine & Li 2007 scaling;
  $L_{\rm dust}$ = rest 8–1000 µm from 9a, $M_{\rm dust}$ = caesar `masses.dust`) is
  still computed, but only as the **9c closure diagnostic**: posterior
  `umin`/`dust.umean` vs ⟨U⟩ now *tests* the mapping instead of assuming it
  (recovering ⟨U⟩ validates the retired anchored grids; landing systematically colder
  is the $L_{\rm dust}^{\rm CIGALE}/L_{\rm dust}^{\rm RT}=0.75$ story). ~11× more
  models per run than the anchored grids (2 qpah × ~11 umin × 4 gamma vs 2 × ~4 × 1
  per stellar model) — still minutes-scale for `pcigale` on 8 cores.
- **attenuation = the measurand, left free**: `dustatt_modified_CF00` with a denser
  16-node `Av_ISM` grid over 0–3 mag; `mu` and `slope_BC` stay at the CF00 defaults but
  `slope_ISM` ∈ {−0.7, −0.48, −0.3} is freed — the mocks' dust is heated by radiation
  absorbed over 4π while the findx0 sightline shows little reddening (median
  $L_{\rm dust}^{\rm CIGALE}/L_{\rm dust}^{\rm RT}=0.75$ in the pinned runs, worst
  fits at `Av_ISM` = 0 → $L_{\rm dust}=0$), and a greyer ISM curve is the only CIGALE
  lever that buys absorbed energy at fixed optical colour. `Av_ISM` is normalized at V,
  so `bayes.attenuation.Av_ISM` keeps the observed column's V-band meaning (only the
  curve *shape* is no longer forced to CF00).
- **`fit_bands`** pins the fit to the 19 observed bands; everything else is still
  predicted.

Run dirs get a `_sfhfile` suffix (`dust_on_findx0_snapNNN_Zs<z>_sfhfile`), so the
parametric `_dw` (delayedbq quantile-grid) and pinned `_Zs<z>` runs survive as
comparison sets. **If the anchored-umin `_sfhfile` grids were already run**, `mv`
those dirs aside before re-executing this cell — it overwrites `pcigale.ini` in
place, and the anchored-vs-free `Av_ISM` shift is a systematic worth keeping;
9c/9e prefer `_sfhfile`, then `_dw`, then plain `_Zs<z>` (the stale
`_Zs*_aNpN` age-quantile dirs — which DO hold results — are explicitly excluded), then
the prior-free baseline. Submit with the printed `sbatch` line; `SKIP_IF_DONE=True`
makes resubmits skip finished runs.

**2026-08-06c (`_sfhfile2`)** — three changes after diagnosing the 13:15 runs:
(a) **age cap**: CIGALE 2025.1 rejects every model with `sfh.age` above the
**Planck18** universe age at the observed z rounded to `redshift_decimals = 2`
(`pdf_analysis/workers.py`); the Planck13 snapshot age exceeded that ceiling
wherever z rounds *up* — snaps 128 (the whole ALMA-C11 anchor), 091 and 096 —
silently masking 20/72 objects ("No suitable model found"). The age is now
capped at the pcigale ceiling and the `sfh.fits` time axis shifted so the
*recent* end survives. (b) **`mu` + `slope_BC` freed**: the surviving fits had
median reduced χ² = 1.32, but the tail (χ² 7–17) locks `Av_ISM = 0` while
under-predicting PACS/SPIRE/ALMA ×3–100 — an unattenuated sightline over
quasi-isotropic dust emission; the birth-cloud channel absorbs energy with
little diffuse reddening. (c) **`smean` arm**: the same grids also run on the
9a 4-sightline-mean photometry; 9e compares the two to split the tail into
anisotropy-limited vs model-limited.

**2026-08-06d (`_sfhfile3`)** — fine low-`Av_ISM` nodes 0.02/0.03. The `_sfhfile2`
runs fit all 72 objects (age bug cured) and findx0-vs-smean χ² is nearly identical
for the tail → model-limited, not anisotropy. The dust-poor snap132 pair was torn
between `Av_ISM = 0` (dust luminosity ~0) and 0.05 (FIR overshoot); the new nodes
open intermediate energy-balance solutions. The CEERS trio (snap091/096, χ² 10–28:
too-blue models, `slope_ISM` pinned at the −0.7 edge, 1–2-member SFH families with
current-mass bias) is left as-is — steeper slopes / mass-loss-corrected SFHs are
the candidate next levers. Grids otherwise unchanged; run order 9b → sbatch →
9c/9e (no 9a rerun).

In [ ]:
# ── Part 9b · CIGALE inputs + run dirs (snapshot x Z-group) + SLURM array ────
# Self-contained after Part 0 + the Part 9a flux FITS + the Part 9a2 smoothed-
# SFH archive (kernel-restart safe). 2026-08-06 scheme: (1) only sources with
# mock ALMA band-6 flux > ALMA_MIN_UJY are fitted; (2) the SFH is INJECTED
# directly — each run's sfhfromfile table carries the members' OWN smoothed
# archaeological SFHs (1 Myr grid, shape-only via normalise=True) and CIGALE
# marginalizes over that family; no parametric SFH grid. M* stays the fitted
# normalization (9c recovery check). The 9a2 delayed+bq fits remain on disk as
# a descriptive record only; (3) caesar stellar Z pins ONE bc03 node per run
# (cg.split_by_metallicity) and the SFR-weighted gas Z restricts the nebular
# zgas grid (<=3 nodes); (4) dl2014 radiation field FREED (2026-08-06b,
# replacing the semi-pinned <U> scheme — the fully pinned runs piled umin at
# the low grid edge and missed the 4 dust bands at 3-6 sigma (MIPS24 overshot
# ~30%, PACS/SPIRE/ALMA undershot x1.5-3, median L_dust(CIGALE)/L_dust(RT) =
# 0.75), and even the <U> anchors tie the FIR shape to the Ldust/(125 Mdust)
# mapping; with the SFH fixed, any FIR mismatch the grid cannot absorb leaks
# into Av through energy balance): umin on a free 11-node ladder, gamma
# freed to 4 nodes (MIPS24 lives in the warm/PDR component), qpah 2 nodes.
# Dust emission never touches the UV/optical, so this reopens no age-dust
# degeneracy; the members' <U> is kept only as the 9c closure diagnostic;
# (5) attenuation stays the free measurand on a denser Av_ISM grid, with
# slope_ISM freed to greyer options — the only CIGALE lever that buys
# absorbed energy (-> FIR luminosity) at fixed optical colour. Av_ISM is
# defined at V, so its meaning survives the freed slope.
# 2026-08-06c (_sfhfile2): (6) the sfhfromfile age is capped at pcigale's OWN
# age-of-universe ceiling — CIGALE 2025.1 (Planck18, pcigale/utils/
# cosmology.py) rejects every model with sfh.age > universe age at the obs z
# rounded to redshift_decimals=2 (pdf_analysis/workers.py); the Planck13 age
# used before exceeded that ceiling wherever z rounds UP (snaps 128/091/096),
# silently masking 20/72 objects incl. the whole ALMA-C11 anchor snapshot.
# The SFH table is time-shifted so the RECENT end survives the cap (see
# _sfh_file); (7) mu + slope_BC freed — the chi2 tail (7-17) locks Av_ISM=0
# yet under-predicts PACS/SPIRE/ALMA x3-100 (an unattenuated sightline over
# quasi-isotropic RT dust emission); the birth-cloud channel absorbs energy
# with little diffuse optical reddening; (8) the same grids also run on the
# 9a sightline-mean ('smean') photometry — isotropized mocks, so 9e can
# split the tail into anisotropy-limited vs model-limited.
from astropy.cosmology import Planck18
from simbanator.sed import cigale as cg
from simbanator.io.simba import Simulation

PCIGALE_CMD    = cg.find_pcigale()   # dedicated conda env; no activation needed
CORES_PER_TASK = 8
PLOT_SEDS      = True
SKIP_IF_DONE   = False
ALMA_MIN_UJY   = 5.0                 # observed sample is ALMA-selected
LSUN_ERG_S     = 3.828e33
print('pcigale:', PCIGALE_CMD)

CIGDIR     = os.path.join(OUTDIR, 'cigale')
CIGRUNS    = os.path.join(OUTDIR, 'cigale_runs')
FLUX_FITS9 = os.path.join(CIGDIR, 'almac11_fluxes_dust_on_findx0.fits')
SFH_H5     = os.path.join(CIGDIR, 'almac11_sfh_smoothed.h5')
SL_TAGS9B  = ('findx0', 'smean')     # observational-like + isotropized

# the 19 fitted bands = exactly the observed information content (CIGALE names
# verified against the installed 2025.1 DB; SVO 'r'/'i' map to CIGALE 'r+'/'i+')
FIT_BANDS = [
    'cfht.megacam.u', 'cfht.megacam.g',
    'subaru.suprime.B', 'subaru.suprime.V', 'subaru.suprime.r+',
    'subaru.suprime.i+', 'subaru.suprime.IB427', 'subaru.suprime.IB464',
    'paranal.vircam.J', 'paranal.vircam.H', 'paranal.vircam.Ks',
    'spitzer.irac.I1', 'spitzer.irac.I2', 'spitzer.irac.I3', 'spitzer.irac.I4',
    'spitzer.mips.24mu', 'herschel.pacs.green', 'herschel.spire.PSW',
    'alma.band6',
]

SED_MODULES = ('sfhfromfile', 'bc03', 'nebular', 'dustatt_modified_CF00',
               'dl2014', 'restframe_parameters', 'redshifting')

# attenuation is the measurand -> denser grid; slope_ISM freed (Av_ISM is
# normalized at V, so bayes.attenuation.Av_ISM keeps the observed column's
# V-band meaning)
AV_ISM_GRID = [0.0, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.55,
               0.7, 0.9, 1.1, 1.4, 1.7, 2.1, 2.6, 3.0]
# 0.02/0.03 (2026-08-06d): the dust-poor chi2 tail was torn between Av=0
# (dust luminosity ~0, FIR pulls -5) and Av=0.05 (FIR overshoot x1.8);
# with mu=0.2 these nodes give Av_BC = Av_ISM*(1/mu-1) ~ 0.08-0.12 —
# a little absorbed energy without diffuse optical reddening
SLOPE_ISM_GRID = [-0.7, -0.48, -0.3]  # CF00 default + greyer ISM curves
MU_GRID       = [0.2, 0.44, 0.7]  # BC share freed (2026-08-06c): low mu lets
                                  # the fit power the FIR from birth clouds
                                  # without reddening the diffuse optical
SLOPE_BC_GRID = [-1.3, -0.7]      # CF00 default + a greyer BC curve
QPAH_GRID   = [2.50, 5.95]  # powderday PAH_frac['usg']=5.86% + one lower
                            # node (pinned runs overshot rest ~17 um by ~30%)
_UOPT       = cg.grid_options('dl2014', 'umin')
# radiation field freed: the FIR shape floats so the 4 IR bands stop
# torquing Av through energy balance; ladder snapped to the dl2014 nodes
UMIN_GRID   = sorted({float(v) for v in cg.nearest_option(
    [0.1, 0.2, 0.35, 0.6, 1.0, 1.7, 3.0, 5.0, 8.0, 12.0, 25.0],
    _UOPT, log=True)})
GAMMA_GRID  = [0.01, 0.02, 0.05, 0.1]   # PDR fraction freed (MIPS24 = the
                                        # warm component the pin overshot)
RUN_SUFFIX  = '_sfhfile3'   # 2026-08-06d generation (fine low-Av nodes);
                            # _sfhfile2 (age cap + mu/slope_BC + smean) and
                            # _sfhfile (freed-<U>) results stay untouched
                            # for the generation-over-generation A/B

ANALYSIS_PARAMS9 = {
    'variables': ['stellar.m_star', 'stellar.age_m_star', 'stellar.metallicity',
                  'sfh.sfr', 'sfh.sfr100Myrs', 'sfh.index',
                  'attenuation.Av_ISM', 'attenuation.Av_BC',
                  'attenuation.generic.bessell.V',
                  'attenuation.generic.bessell.B',
                  'dust.luminosity', 'dust.mass', 'dust.umean',
                  'param.restframe_generic.johnson.U-generic.johnson.V',
                  'param.restframe_generic.johnson.V-generic.johnson.J'],
    'save_best_sed': True,
}

# ── per-galaxy smoothed SFHs (9a2 archive) ──
if not os.path.exists(SFH_H5):
    raise RuntimeError(f'{SFH_H5} missing — re-run Part 9a2 first (9b injects '
                       'the smoothed SIMBA SFHs via sfhfromfile)')
SFH_ARCH = {}                       # id -> (t_gyr, sfr_msun_yr)
with h5py.File(SFH_H5, 'r') as f5:
    for k_ in f5:
        d_ = np.asarray(f5[k_][:], float)
        SFH_ARCH[str(k_)] = (d_[:, 0], d_[:, 1])
print(f'[sfh] smoothed archive: {len(SFH_ARCH)} galaxies from {SFH_H5}')

def _sfh_file(rd, ids, age_myr, shift_myr=0):
    # write rd/sfh.fits for sfhfromfile: col 0 = time [Myr] 0..age_myr in
    # strict 1 Myr steps (CIGALE raises otherwise), one SFR column per member
    # with an archived SFH. Interp from the 25 Myr smoothed grid; SFR = 0
    # before the first archived sample, edge-held after the last one (the
    # recent edge drives the UV/nebular fluxes). normalise=True in the module
    # -> only the shape matters. shift_myr > 0 when the pcigale age ceiling
    # is below the Planck13 snapshot age: table time t maps to cosmic time
    # t + shift_myr, i.e. the OLDEST shift_myr are dropped and t = age_myr
    # still lands on the snapshot epoch (sfhfromfile truncates at the recent
    # end via time <= age, so shrinking age alone would cut the newest SFR).
    # Returns the member ids actually written.
    t1 = np.arange(int(age_myr) + 1)                        # Myr, step 1
    tab = Table()
    tab['time'] = t1.astype(np.int64)
    kept = []
    for i_ in ids:
        arch = SFH_ARCH.get(str(i_))
        if arch is None:
            continue
        ts_, ss_ = arch
        sfr1 = np.interp(t1 + shift_myr, ts_ * 1e3, ss_, left=0.0)
        tab[str(i_)] = np.clip(sfr1, 0.0, None)             # right: edge-held
        kept.append(str(i_))
    if not kept:
        return None
    os.makedirs(rd, exist_ok=True)
    tab.write(os.path.join(rd, 'sfh.fits'), overwrite=True)
    return kept

# ── input files: full catalog -> ALMA gate -> NaN-error repair, per tag ──
gated9 = {}
for tag_ in SL_TAGS9B:
    ff_ = os.path.join(CIGDIR, f'almac11_fluxes_dust_on_{tag_}.fits')
    if not os.path.exists(ff_):
        print(f'[{tag_}] {os.path.basename(ff_)} missing — re-run Part 9a '
              f'first; tag skipped')
        continue
    data_all = cg.write_cigale_input(
        ff_, os.path.join(CIGDIR, f'cigale_dust_on_{tag_}.fits'))
    t9 = Table.read(data_all)
    _alma9 = 1e3 * np.asarray(t9['alma.band6'], float)       # mJy -> uJy
    _keep9 = np.isfinite(_alma9) & (_alma9 > ALMA_MIN_UJY)
    print(f'[gate:{tag_}] ALMA band 6 > {ALMA_MIN_UJY:g} uJy: '
          f'{int(_keep9.sum())}/{len(t9)} sources fitted')
    t9 = t9[_keep9]
    _bands9 = [c for c in t9.colnames
               if c not in ('id', 'redshift', 'distance')
               and not c.endswith('_err')]
    n_fix = 0
    for b_ in _bands9:
        f_ = np.asarray(t9[b_], float)
        e_ = np.asarray(t9[f'{b_}_err'], float)
        bad = np.isfinite(f_) & ~np.isfinite(e_)
        if bad.any():
            t9[f'{b_}_err'][bad] = 0.1 * np.abs(f_[bad])
            n_fix += int(bad.sum())
    t9.write(data_all, overwrite=True)   # gated (+repaired) catalog on disk
    if n_fix:
        print(f'[sanitize:{tag_}] {n_fix} NaN error(s) -> 10% of the flux '
              f'(CIGALE adds additionalerror=0.1 in quadrature on top)')
    gated9[tag_] = t9
if not gated9:
    raise RuntimeError('no flux tables found — run Part 9a first')

_snaps_all9 = sorted({int(re.match(r'snap(\d+)_', str(i_)).group(1))
                      for t_ in gated9.values() for i_ in t_['id']})

# ── SIMBA truth per galaxy: caesar Z_star / Z_gas^SFR / M_dust ──
ZS_MAP, ZG_MAP, MD_MAP = {}, {}, {}
_sim9 = Simulation('cis100')
for snap in _snaps_all9:
    with h5py.File(_sim9.get_caesar_file(int(snap)), 'r') as fc_:
        d_ = fc_['galaxy_data']
        for g_, zs_, zg_, md_ in zip(
                np.asarray(d_['GroupID'][:], int),
                np.asarray(d_['dicts/metallicities.stellar'][:], float),
                np.asarray(d_['dicts/metallicities.sfr_weighted'][:], float),
                np.asarray(d_['dicts/masses.dust'][:], float)):
            k_ = f'snap{int(snap):03d}_gal{int(g_)}'
            ZS_MAP[k_] = float(zs_) if zs_ > 0 else np.nan
            ZG_MAP[k_] = float(zg_) if zg_ > 0 else np.nan
            MD_MAP[k_] = float(md_) if md_ > 0 else np.nan

# <U> = Ldust / (125 Mdust)  (Draine & Li 2007 scaling, solar units) —
# closure DIAGNOSTIC only since 2026-08-06b: the grid no longer uses it;
# 9c compares the posterior umin / dust.umean against it instead
_tf9 = Table.read(FLUX_FITS9)
UMEAN = {}
if 'Ldust_8_1000_Lsun' not in _tf9.colnames:
    print('[dl2014] flux FITS lacks Ldust_8_1000_Lsun — <U> closure '
          'diagnostic unavailable (the free grid does not need it)')
else:
    for r in _tf9:
        k_ = f"snap{int(r['snap']):03d}_gal{int(r['gal_id_at_snap'])}"
        ld_, md_ = float(r['Ldust_8_1000_Lsun']), MD_MAP.get(k_, np.nan)
        UMEAN[k_] = (ld_ / (125.0 * md_)
                     if np.isfinite(ld_) and np.isfinite(md_) and md_ > 0
                     else np.nan)
_uf = np.array([v for v in UMEAN.values() if np.isfinite(v)], float)
print('[dl2014] <U> = Ldust/(125 Mdust) diagnostic: '
      + (' / '.join(f'{v:.2f}' for v in np.percentile(_uf, [5, 50, 95]))
         + ' (p5/p50/p95)' if _uf.size else 'n/a')
      + f'\n[dl2014] free grid: umin {UMIN_GRID} | gamma {GAMMA_GRID} | '
      + f'qpah {QPAH_GRID}\n[cf00] free grid: mu {MU_GRID} | '
      + f'slope_BC {SLOPE_BC_GRID}')

_ing = [str(i_) for i_ in gated9[next(iter(gated9))]['id']]
print(f'[sfh] archived SFHs for '
      f'{sum(i_ in SFH_ARCH for i_ in _ing)}/{len(_ing)} gated galaxies | '
      f'caesar Z for {sum(np.isfinite(ZS_MAP.get(i_, np.nan)) for i_ in _ing)}')

run_dirs9 = []
for tag_, t9 in gated9.items():
    _snap9 = np.array([int(re.match(r'snap(\d+)_', str(i_)).group(1))
                       for i_ in t9['id']])
    for snap in sorted(set(_snap9.tolist())):
        sub = t9[_snap9 == snap]
        z = float(np.median(np.asarray(sub['redshift'], float)))
        age13 = int(round(Planck13.age(z).to(u.Myr).value))
        # pcigale rejects sfh.age > Planck18 universe age at the obs z
        # rounded to redshift_decimals=2; take the strictest ceiling - 1 Myr
        age_cig = int(np.floor(min(
            Planck18.age(z).to(u.Myr).value,
            Planck18.age(round(z, 2)).to(u.Myr).value))) - 1
        age_myr = min(age13, age_cig)
        df = os.path.join(CIGDIR,
                          f'cigale_dust_on_{tag_}_snap{snap:03d}.fits')
        sub.write(df, overwrite=True)
        for grp in cg.split_by_metallicity(df, ZS_MAP, zgas=ZG_MAP):
            gt = Table.read(grp['path'])
            ids_ = [str(i_) for i_ in gt['id']]
            rd = os.path.join(
                CIGRUNS, f'dust_on_{tag_}_snap{snap:03d}'
                + (f"_{grp['tag']}" if grp['tag'] else '') + RUN_SUFFIX)
            kept = _sfh_file(rd, ids_, age_myr, shift_myr=age13 - age_myr)
            if kept is None:
                print(f'  [skip] {os.path.basename(rd)}: none of the '
                      f'{len(ids_)} member(s) has an archived SFH')
                continue
            if len(kept) < len(ids_):
                print(f'  [note] {os.path.basename(rd)}: '
                      f'{len(ids_) - len(kept)} member(s) without archived '
                      f'SFH — family = {len(kept)} SFHs')
            mp = {
                'sfhfromfile': {'filename': os.path.join(rd, 'sfh.fits'),
                                'sfr_column': list(range(1, len(kept) + 1)),
                                'age': [age_myr], 'normalise': True},
                'bc03': {'imf': 1, 'metallicity': [0.008, 0.02, 0.05]},
                'dustatt_modified_CF00': {'Av_ISM': AV_ISM_GRID,
                                          'slope_ISM': SLOPE_ISM_GRID,
                                          'mu': MU_GRID,
                                          'slope_BC': SLOPE_BC_GRID},
                'dl2014': {'qpah': QPAH_GRID, 'umin': UMIN_GRID,
                           'gamma': GAMMA_GRID},
                'restframe_parameters': {
                    'colours_filters': ('generic.johnson.U-generic.johnson.V'
                                        ' & '
                                        'generic.johnson.V-generic.johnson.J')},
            }
            for k_, v_ in (grp['module_params'] or {}).items():
                mp[k_] = {**mp.get(k_, {}), **v_}     # bc03 Z pin + zgas nodes
            cg.prepare_run(rd, grp['path'], sed_modules=SED_MODULES,
                           module_params=mp, analysis_params=ANALYSIS_PARAMS9,
                           cores=CORES_PER_TASK, fit_bands=FIT_BANDS)
            run_dirs9.append(rd)
            print(f"  {os.path.basename(rd):40s} n={len(gt):3d} | "
                  f"SFH family {len(kept)} | age {age_myr} Myr"
                  + (f' (Planck13 {age13} capped)' if age_myr < age13 else ''))

job9 = cg.write_slurm_array(run_dirs9,
                            os.path.join(CIGRUNS, 'cigale_almac11.job'),
                            pcigale_cmd=PCIGALE_CMD, cores=CORES_PER_TASK,
                            plots=PLOT_SEDS, job_name='cigale_almac11',
                            skip_if_done=SKIP_IF_DONE)
print(f'\n{len(run_dirs9)} run dir(s) prepared. Submit with:\n'
      f'  sbatch {job9}\n'
      f'(add -p <partition> if needed; logs in '
      f'{os.path.join(CIGRUNS, "slurm_logs")})')

### Part 9c — after the fits: consistent Av_ISM comparison + estimator closure

Run once the array has drained (`ls cigale_runs/dust_on_findx0_snap*/out/results.fits`).
Needs Part 3 in-session (`attenuation_curve`). Auto-detects the prior-informed runs
(`*_Zs*_dw` first, then plain `*_Zs<z>`; the stale `*_Zs*_aNpN` age-quantile dirs are
excluded — they hold results and used to shadow the newer fits) and falls back to the
prior-free ones when none are fitted yet; when both exist, the median `Av_ISM` shift
between them is printed. Three figures + a per-target
table, all saved:

- **like-for-like** — the Part 3 strip plot redone with the mock `bayes.attenuation.Av_ISM`
  (green) against the observed `Av_ISM` (black); the RT $A_V$ medians (dashed blue) show how
  much of the original discrepancy was definitional.
- **estimator closure** — per galaxy, CIGALE's total V attenuation
  (`attenuation.generic.bessell.V` ≡ observed `V_B90`, orange) and `Av_ISM` (green) against
  the RT truth. The orange series answers "does CIGALE recover the true attenuation from 19
  bands?"; the green one shows what fraction the ISM component captures.
- **truth recovery** — recovered vs caesar $\log M_*$ and mass-weighted stellar age (from
  the `rt_union.fits` UNION columns). With Z pinned and the age floor set by the priors,
  any residual mass/age bias bounds how much of a remaining $A_V$ discrepancy can still be
  an SFH artefact.

The joined per-galaxy table lands in `cigale/almac11_cigale_av_comparison.fits`.

In [ ]:
# ── Part 9c · consistent A_V: CIGALE(mock) vs CIGALE(obs) vs RT truth ────────
# Run AFTER the job array drains. Needs Part 0 (attenuation_curve).
from simbanator.sed.cigale import nmad

CIGRUNS = os.path.join(OUTDIR, 'cigale_runs')
CIGDIR  = os.path.join(OUTDIR, 'cigale')
_zs_tail = lambda d_: os.path.basename(d_).split('_Zs', 1)[-1]
_rdirs = sorted(glob.glob(os.path.join(CIGRUNS,
                                       'dust_on_findx0_snap*_Zs*_sfhfile3')))
if not _rdirs:      # _sfhfile2 gen (age cap + mu/slope_BC, coarse low-Av)
    _rdirs = sorted(glob.glob(os.path.join(CIGRUNS,
                                           'dust_on_findx0_snap*_Zs*_sfhfile2')))
if not _rdirs:      # previous injected-SFH gen (Planck13 age, mu fixed)
    _rdirs = sorted(glob.glob(os.path.join(CIGRUNS,
                                           'dust_on_findx0_snap*_Zs*_sfhfile')))
if not _rdirs:      # older parametric-SFH (delayedbq quantile-grid) runs
    _rdirs = sorted(glob.glob(os.path.join(CIGRUNS,
                                           'dust_on_findx0_snap*_Zs*_dw')))
if not _rdirs:      # older pinned-dl2014 runs: keep ONLY plain _Zs<z> dirs
    _rdirs = [d_ for d_ in sorted(glob.glob(os.path.join(
        CIGRUNS, 'dust_on_findx0_snap*_Zs*')))
        if '_' not in _zs_tail(d_)]   # drops the dead _Zs*_aNpN dirs too
_prior_runs = bool(_rdirs)
if not _prior_runs:                     # fall back to the prior-free 9b runs
    _rdirs = sorted(glob.glob(os.path.join(
        CIGRUNS, 'dust_on_findx0_snap[0-9][0-9][0-9]')))
CFIT, _miss9 = {}, []
for rd in _rdirs:
    f_ = os.path.join(rd, 'out', 'results.fits')
    if not os.path.exists(f_):
        _miss9.append(os.path.basename(rd)); continue
    for r in Table.read(f_):
        CFIT[str(r['id'])] = r
print(f'{len(CFIT)} fitted galaxies from {len(_rdirs) - len(_miss9)}/{len(_rdirs)} runs'
      + (f' — NOT finished: {_miss9}' if _miss9 else ''))

# prior-free baseline (the original 9b run dirs, no _Zs suffix): report how
# far the SIMBA-truth priors moved the attenuation solution
CFIT0 = {}
if _prior_runs:
    for rd in sorted(glob.glob(os.path.join(
            CIGRUNS, 'dust_on_findx0_snap[0-9][0-9][0-9]'))):
        f_ = os.path.join(rd, 'out', 'results.fits')
        if os.path.exists(f_):
            for r in Table.read(f_):
                CFIT0[str(r['id'])] = r
    if CFIT0:
        _dav = np.array([float(CFIT[i_]['bayes.attenuation.Av_ISM'])
                         - float(CFIT0[i_]['bayes.attenuation.Av_ISM'])
                         for i_ in CFIT if i_ in CFIT0])
        print(f'[priors] Av_ISM shift vs the prior-free baseline: median '
              f'{np.median(_dav):+.3f} mag, NMAD {nmad(_dav):.3f} mag '
              f'({_dav.size} galaxies)')

_AVK = 'bayes.attenuation.generic.bessell.V'   # total V attenuation == obs V_B90
_jV9 = int(np.argmin(np.abs(GRID - 0.55)))
rows9c = []
for tid in SELECTED:
    for r in SELECTED[tid]:
        snap, gid = int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])
        c_ = CFIT.get(f'snap{snap:03d}_gal{gid}')
        if c_ is None:
            continue
        curve = attenuation_curve(snap, gid)
        rows9c.append(dict(
            tid=tid, id=f'snap{snap:03d}_gal{gid}',
            av_rt=np.nan if curve is None else float(curve[_jV9]),
            av_ism=float(c_['bayes.attenuation.Av_ISM']),
            av_ism_err=float(c_['bayes.attenuation.Av_ISM_err']),
            av_tot=float(c_[_AVK]), av_tot_err=float(c_[f'{_AVK}_err']),
            chi2=float(c_['best.reduced_chi_square'])))
T9C = Table(rows9c)
T9C.write(os.path.join(CIGDIR, 'almac11_cigale_av_comparison.fits'),
          overwrite=True)
print(f'{len(T9C)} (target, member) pairs | median reduced chi2 = '
      f'{np.median(np.asarray(T9C["chi2"], float)):.2f}')

# ── figure 1: the Part 3 comparison, now like-for-like ──
_t9 = [t for t in TARGETS if TARGETS[t]['sample'] == 'almac11'
       and np.isfinite(TARGETS[t]['av_ism']) and t in SELECTED]
_lab9 = {'ism': 'mock CIGALE Av_ISM', 'rt': r'RT $A_V$ (Part 3)'}
fig, ax = plt.subplots(figsize=(10.5, 5.2))
for x, tid in enumerate(_t9):
    S = T9C[np.asarray(T9C['tid']) == tid]
    if len(S) == 0:
        continue
    y_ism = np.asarray(S['av_ism'], float)
    ax.plot(np.full(y_ism.size, x) + np.random.uniform(-0.15, 0.15, y_ism.size),
            y_ism, 'o', ms=3, color='tab:green', alpha=0.45, mec='none')
    ax.hlines(np.median(y_ism), x - 0.28, x + 0.28, color='tab:green', lw=2.2,
              label=_lab9.pop('ism', None))
    y_rt = np.asarray(S['av_rt'], float)
    y_rt = y_rt[np.isfinite(y_rt)]
    if y_rt.size:
        ax.hlines(np.median(y_rt), x - 0.28, x + 0.28, color='tab:blue',
                  lw=1.4, ls='--', label=_lab9.pop('rt', None))
    ax.errorbar(x, TARGETS[tid]['av_ism'], yerr=TARGETS[tid]['av_ism_err'],
                fmt='s', ms=6, color='k', capsize=3, **obs_mstyle(tid))
ax.set(xticks=range(len(_t9)), xticklabels=_t9, ylabel=r'$A_V$ [mag]',
       title='consistent extraction: mock CIGALE Av_ISM (green) vs observed '
             '(black; hollow = control)')
ax.tick_params(axis='x', rotation=60)
ax.grid(alpha=0.2, axis='y', lw=0.5)
ax.legend(handles=ax.get_legend_handles_labels()[0] + obs_proxy(),
          fontsize=9, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'av_vs_cigale_consistent.png'), dpi=180)
plt.show()

# ── figure 2: estimator closure — what does CIGALE do to the RT truth? ──
m9 = np.isfinite(np.asarray(T9C['av_rt'], float))
x9 = np.asarray(T9C['av_rt'], float)[m9]
fig, ax = plt.subplots(figsize=(6.4, 5.6))
lim9 = (0.0, max(1.05 * np.nanmax(x9), 1.0))
ax.plot(lim9, lim9, color='0.6', lw=0.8)
for key, c_, lab in (('av_tot', 'tab:orange',
                      r'total (attenuation.generic.bessell.V $\equiv$ V_B90)'),
                     ('av_ism', 'tab:green', 'Av_ISM (ISM component)')):
    y9 = np.asarray(T9C[key], float)[m9]
    ax.plot(x9, y9, 'o', ms=4, color=c_, alpha=0.55, mec='none', label=lab)
    d9 = (y9 - x9)[np.isfinite(y9 - x9)]
    print(f'{lab:55s}: median offset {np.nanmedian(d9):+.2f} mag, '
          f'NMAD {nmad(d9):.2f} mag')
ax.set(xlim=lim9, ylim=(0, None),
       xlabel=r'RT truth: $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ [mag]',
       ylabel='CIGALE estimate [mag]', title='estimator closure on the mocks')
ax.grid(alpha=0.2, lw=0.5)
ax.legend(fontsize=8.5, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'av_cigale_closure.png'), dpi=180)
plt.show()

# ── per-target summary ──
print(f"\n{'tid':>8s} {'N':>3s} {'RT A_V':>7s} {'mock ISM':>8s} "
      f"{'mock tot':>8s} {'obs ISM':>7s}")
for tid in _t9:
    S = T9C[np.asarray(T9C['tid']) == tid]
    if len(S) == 0:
        continue
    print(f"{tid:>8s} {len(S):3d} "
          f"{np.nanmedian(np.asarray(S['av_rt'], float)):7.2f} "
          f"{np.median(np.asarray(S['av_ism'], float)):8.2f} "
          f"{np.median(np.asarray(S['av_tot'], float)):8.2f} "
          f"{TARGETS[tid]['av_ism']:7.2f}")

# ── figure 3: truth recovery — the priors' sanity check ──
# Z_star and the age floor are pinned per run; M* is NOT (it is the fitted
# normalization), so recovered-vs-caesar mass is the meaningful closure test,
# and the mass-weighted age shows what the age floor left free.
_tru = {(int(r_['SNAPSHOT']), int(r_['GROUPID_SNAPSHOT'])):
        (float(r_['LOG_MSTAR']), float(r_['AGE_STAR'])) for r_ in UNION}
_rec = []
for (snap_, gid_), (lm_, ag_) in _tru.items():
    c_ = CFIT.get(f'snap{snap_:03d}_gal{gid_}')
    if c_ is None:
        continue
    _rec.append((lm_, np.log10(max(float(c_['bayes.stellar.m_star']), 1.0)),
                 ag_, float(c_['bayes.stellar.age_m_star']) / 1e3))  # Myr->Gyr
_rec = np.array(_rec)
if _rec.size:
    fig, axs = plt.subplots(1, 2, figsize=(10.6, 4.9))
    for ax, jx, jy, lab, unit in (
            (axs[0], 0, 1, r'$\log\,M_*$', 'dex'),
            (axs[1], 2, 3, 'mass-weighted age', 'Gyr')):
        x_, y_ = _rec[:, jx], _rec[:, jy]
        lim = (min(x_.min(), np.nanmin(y_)), max(x_.max(), np.nanmax(y_)))
        ax.plot(lim, lim, color='0.6', lw=0.8)
        ax.plot(x_, y_, 'o', ms=4, color='tab:purple', alpha=0.5, mec='none')
        d_ = y_ - x_
        ax.set(xlim=lim, ylim=lim, xlabel=f'SIMBA truth {lab}',
               ylabel=f'CIGALE {lab}',
               title=f'{lab}: median {np.nanmedian(d_):+.2f} {unit}, '
                     f'NMAD {nmad(d_[np.isfinite(d_)]):.2f} {unit}')
        ax.grid(alpha=0.2, lw=0.5)
    fig.suptitle('truth recovery on the mocks (priors pin Z + age floor; '
                 'M* is fitted)', y=1.03)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, 'cigale_truth_recovery.png'), dpi=180,
                bbox_inches='tight')
    plt.show()

### Part 9d — CIGALE $A_V$ vs SIMBA $A_V$ + Part 7 with the CIGALE $A_V$

Run after 9c (`CFIT`). Two figures, both on the deduped Part-7 sample (union of
`SELECTED` members with a CIGALE fit):

- **CIGALE vs SIMBA, per galaxy** — mock CIGALE `Av_ISM` (green) and total V attenuation
  `attenuation.generic.bessell.V` (orange) against the SIMBA RT
  $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ at `findx0`; AGN hosts as triangles. The
  per-galaxy view behind 9c's closure medians: where CIGALE lands relative to the
  simulation truth, and whether AGN hosts bias differently.
- **Part 7 like-for-like** — the dust-ratio panels redone with the mock CIGALE `Av_ISM`
  on the y-axis, so the simulated points and the observed black squares are the *same
  estimator quantity* (the original Part 7 mixed the RT $A_V$ with the observed
  `Av_ISM`). Spearman printed for both `Av_ISM` and the total V attenuation.

The CIGALE fits use only `findx0` photometry, so there is no per-sightline CIGALE $A_V$ —
the sightline-resolved dust-mass figure lives in Part 8 with the RT $A_V$. Depends on
Parts 3 (`attenuation_curve`), 5 (`LOGMD_OBS`) and 9c (`CFIT`).


In [ ]:
# ── Part 9d · CIGALE A_V vs SIMBA RT A_V + Part 7 redone with CIGALE A_V ─────
# Run AFTER 9c (CFIT). Needs Parts 0 (attenuation_curve) + 5 (LOGMD_OBS).
try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None
from simbanator.sed.cigale import nmad

_AVK9d = 'bayes.attenuation.generic.bessell.V'
_jV9d = int(np.argmin(np.abs(GRID - 0.55)))
_rows9d = {}                               # (snap,gid) -> selection row, deduped
for tid in SELECTED:
    for r in SELECTED[tid]:
        _rows9d.setdefault((int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])), r)
_recs = []
for (snap, gid), r in _rows9d.items():
    c_ = CFIT.get(f'snap{snap:03d}_gal{gid}')
    if c_ is None:
        continue
    curve = attenuation_curve(snap, gid)
    md, mg = float(r['MDUST']), float(r['MGAS'])
    _recs.append((
        np.nan if curve is None else float(curve[_jV9d]),
        float(c_['bayes.attenuation.Av_ISM']), float(c_[_AVK9d]),
        np.log10(md) - float(r['LOG_MSTAR']) if md > 0 else np.nan,
        np.log10(md / mg) if (md > 0 and mg > 0) else np.nan,
        str(r['AGN_CLASS']) == 'AGN'))
assert _recs, 'no CIGALE-fitted galaxies in SELECTED — run 9b, then 9c, first'
av_rt, av_ism, av_tot, ds9, dg9, isagn9 = map(np.asarray, zip(*_recs))
isagn9 = isagn9.astype(bool)
print(f'{len(_recs)}/{len(_rows9d)} selected galaxies have a CIGALE fit')

# ── figure 1: per-galaxy CIGALE A_V vs SIMBA RT A_V ──
m9d = np.isfinite(av_rt)
fig, ax = plt.subplots(figsize=(6.4, 5.6))
lim = (0.0, max(1.0, 1.05 * np.nanmax(np.concatenate(
    [av_rt[m9d], av_ism, av_tot]))))
ax.plot(lim, lim, color='0.6', lw=0.8)
for y_, c_, lab in ((av_tot, 'tab:orange', 'total V (bessell.V)'),
                    (av_ism, 'tab:green', 'Av_ISM')):
    for flag, mk in ((False, 'o'), (True, '^')):
        mm = m9d & (isagn9 == flag)
        ax.plot(av_rt[mm], y_[mm], mk, ms=4.5, color=c_, alpha=0.55, mec='none',
                label=lab + (', AGN host' if flag else ''))
    d_ = (y_ - av_rt)[m9d & np.isfinite(y_)]
    print(f'{lab:22s}: median CIGALE - RT offset {np.median(d_):+.2f} mag, '
          f'NMAD {nmad(d_):.2f} mag')
ax.set(xlim=lim, ylim=(0, None),
       xlabel=r'SIMBA RT $A_V$ [mag] (findx0)', ylabel='CIGALE estimate [mag]',
       title='CIGALE $A_V$ vs SIMBA RT $A_V$ (Part 7 sample)')
ax.grid(alpha=0.2, lw=0.5)
ax.legend(fontsize=8.5, frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'av_cigale_vs_simba.png'), dpi=180)
plt.show()

# ── figure 2: Part 7 like-for-like — dust ratios vs the CIGALE Av_ISM ──
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)
for flag, c_, lab in ((False, 'tab:blue', 'non-AGN host'),
                      (True, 'tab:orange', 'AGN host')):
    m = isagn9 == flag
    axes[0].plot(ds9[m], av_ism[m], 'o', ms=4, color=c_, alpha=0.55, mec='none',
                 label=lab)
    axes[1].plot(dg9[m], av_ism[m], 'o', ms=4, color=c_, alpha=0.55, mec='none')
for tid, T in TARGETS.items():             # observed side — as Part 7
    if T['row'] is None or not np.isfinite(T['av_ism']):
        continue
    if tid in LOGMD_OBS and np.isfinite(LOGMD_OBS[tid]):
        axes[0].errorbar(LOGMD_OBS[tid] - T['logm'], T['av_ism'], yerr=T['av_ism_err'],
                         fmt='s', ms=6, color='k', capsize=2, zorder=5,
                         **obs_mstyle(tid))
    dgr = float(T['row'].get('logDGR', np.nan))
    if np.isfinite(dgr):
        axes[1].errorbar(dgr, T['av_ism'], yerr=T['av_ism_err'], fmt='s', ms=6,
                         color='k', capsize=2, zorder=5, **obs_mstyle(tid))
axes[1].axvline(-3, color='0.6', ls='--', lw=0.8)      # dusty-selection threshold
axes[1].axvline(-4, color='0.6', ls=':', lw=0.8)       # control threshold
axes[0].set(xlabel=r'$\log\,M_{\rm dust}/M_\star$',
            ylabel='Av_ISM [mag] — mock CIGALE (colored) / obs CIGALE (black)')
axes[1].set(xlabel=r'sim: $\log\,M_{\rm dust}/M_{\rm gas,tot}$   ·   '
                   r'obs: logDGR ($M_{\rm dust}/M_{\rm mol}$)')
for ax in axes:
    ax.grid(alpha=0.2, lw=0.5)
axes[0].legend(handles=axes[0].get_legend_handles_labels()[0] + obs_proxy(),
               frameon=False, fontsize=9)
fig.suptitle('Part 7 like-for-like: dust ratios vs the CIGALE Av_ISM')
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'dust_ratio_vs_av_cigale.png'), dpi=180,
            bbox_inches='tight')
plt.show()

for yname, y_ in (('Av_ISM', av_ism), ('total V', av_tot)):
    for xname, x_ in (('log Md/M*', ds9), ('log Md/Mgas', dg9)):
        m = np.isfinite(x_) & np.isfinite(y_)
        if spearmanr is not None and m.sum() > 3:
            rho, p = spearmanr(x_[m], y_[m])
            print(f'Spearman CIGALE {yname} vs {xname}: rho={rho:+.2f} '
                  f'(p={p:.1e}, N={m.sum()})')


### Part 9e — fit quality vs dust content: is a poor $\chi^2$ a dust-poor analogue?

Theory: analogues with a low dust fraction produce no significant FIR/sub-mm emission
in powderday, so the FIR half of the mock photometry (MIPS 24, PACS, SPIRE, ALMA band 6)
gives CIGALE nothing to anchor its dust model on and the fit degrades. Plots the caesar
$M_{\rm dust}$, $M_{\rm dust}/M_\star$, $M_{\rm H_2}/M_\star$ and
$M_{\rm dust}/M_{\rm H_2}$ against the CIGALE reduced $\chi^2$ of the mock dust_on
fits, colored by the mock ALMA band-6 flux CIGALE actually fitted (rest-frame
$\sim$0.9 mm at snaps 128–132).

Since 2026-08-06c this cell also joins the findx0 runs against the
sightline-mean (`smean`) runs: the mock is isotropized there, so tail objects
whose χ² collapses under `smean` are anisotropy-limited (CIGALE's isotropic
energy balance vs a single RT sightline), not model-limited
(`figures/chi2_findx0_vs_smean.png`).

In [ ]:
# ── Part 9e · fit quality vs dust content — does missing dust break the fit? ─
# Theory test: analogues with little dust produce no significant FIR/sub-mm in
# powderday, so the FIR half of the 19-band mock photometry (MIPS24, PACS,
# SPIRE, ALMA) carries no dust signal and CIGALE cannot anchor its dust model
# -> poor reduced chi2. Needs only Part 0 (reuses the 9c CFIT if that already
# ran, else loads the CIGALE results here). Dust/H2/stellar
# masses come straight from caesar; the ALMA band-6 flux is the Part 9a mock
# flux CIGALE actually fitted (observed-frame 1.3 mm -> rest ~0.9 mm at snaps
# 128-132, ~0.55 mm at the CEERS snaps 091/096).
from simbanator.io.simba import Simulation
try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

CIGRUNS = os.path.join(OUTDIR, 'cigale_runs')
CIGDIR  = os.path.join(OUTDIR, 'cigale')
if 'CFIT' not in globals():             # standalone: same loader as Part 9c
    _zs_tail9e = lambda d_: os.path.basename(d_).split('_Zs', 1)[-1]
    _rdirs = (sorted(glob.glob(os.path.join(CIGRUNS,
                                            'dust_on_findx0_snap*_Zs*_sfhfile3')))
              or sorted(glob.glob(os.path.join(CIGRUNS,
                                               'dust_on_findx0_snap*_Zs*_sfhfile2')))
              or sorted(glob.glob(os.path.join(CIGRUNS,
                                               'dust_on_findx0_snap*_Zs*_sfhfile')))
              or sorted(glob.glob(os.path.join(CIGRUNS,
                                               'dust_on_findx0_snap*_Zs*_dw')))
              or [d_ for d_ in sorted(glob.glob(os.path.join(
                      CIGRUNS, 'dust_on_findx0_snap*_Zs*')))
                  if '_' not in _zs_tail9e(d_)]   # skip dead _Zs*_aNpN dirs
              or sorted(glob.glob(os.path.join(
                  CIGRUNS, 'dust_on_findx0_snap[0-9][0-9][0-9]'))))
    CFIT = {}
    for rd in _rdirs:
        f_ = os.path.join(rd, 'out', 'results.fits')
        if os.path.exists(f_):
            for r in Table.read(f_):
                CFIT[str(r['id'])] = r
    print(f'[9e] CFIT loaded standalone: {len(CFIT)} galaxies from '
          f'{len(_rdirs)} run dirs')

_sim9e = Simulation('cis100')
_mass9e = {}                                 # (snap,gid) -> (Mdust, MH2, M*)
for snap in sorted({int(k[4:7]) for k in CFIT}):
    with h5py.File(_sim9e.get_caesar_file(snap), 'r') as fc_:
        d_ = fc_['galaxy_data']
        for g_, md_, mh2_, ms_ in zip(
                np.asarray(d_['GroupID'][:], int),
                np.asarray(d_['dicts/masses.dust'][:], float),
                np.asarray(d_['dicts/masses.H2'][:], float),
                np.asarray(d_['dicts/masses.stellar'][:], float)):
            _mass9e[(int(snap), int(g_))] = (float(md_), float(mh2_), float(ms_))

_f9e = Table.read(os.path.join(CIGDIR, 'almac11_fluxes_dust_on_findx0.fits'))
_alma9e = {(int(r['snap']), int(r['gal_id_at_snap'])):
           float(r['ALMA.ALMA.band6']) for r in _f9e}
_agn9e = {(int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT'])): bool(r['AGN_ANY'])
          for r in UNION}

_lg9e = lambda v: np.log10(v) if np.isfinite(v) and v > 0 else np.nan
rows9e = []
for key, c_ in CFIT.items():
    snap, gid = int(key[4:7]), int(key.split('_gal')[1])
    if (snap, gid) not in _mass9e:
        continue
    md, mh2, ms = _mass9e[(snap, gid)]
    rows9e.append(dict(
        snap=snap, gid=gid, chi2=float(c_['best.reduced_chi_square']),
        logmd=_lg9e(md),
        ds=_lg9e(md / ms) if ms > 0 else np.nan,
        h2s=_lg9e(mh2 / ms) if ms > 0 else np.nan,
        d2h2=_lg9e(md / mh2) if mh2 > 0 else np.nan,
        salma=_lg9e(_alma9e.get((snap, gid), np.nan)),
        agn=_agn9e.get((snap, gid), False)))
T9E = Table(rows9e)
T9E.write(os.path.join(CIGDIR, 'almac11_chi2_dustcontent.fits'), overwrite=True)
chi9e = np.asarray(T9E['chi2'], float)
sal9e = np.asarray(T9E['salma'], float)
agn9e = np.asarray(T9E['agn'], bool)
print(f'{len(T9E)} fitted galaxies | no caesar match: '
      f'{len(CFIT) - len(T9E)} | no ALMA mock flux: '
      f'{int((~np.isfinite(sal9e)).sum())} | MH2=0: '
      f'{int((~np.isfinite(np.asarray(T9E["d2h2"], float))).sum())}')

import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
fal9e = 1e3 * 10 ** sal9e                 # mock ALMA band-6 flux in micro-Jy
_fmt9e = mticker.FuncFormatter(lambda v, _p: f'{v:g}')   # plain numbers, no 10^x

# ── figure 1: the four dust/gas quantities vs reduced chi2 ──
_pan9e = (('logmd', r'$\log\,M_{\rm dust}\,[M_\odot]$'),
          ('ds',    r'$\log\,M_{\rm dust}/M_\star$'),
          ('h2s',   r'$\log\,M_{\rm H_2}/M_\star$'),
          ('d2h2',  r'$\log\,M_{\rm dust}/M_{\rm H_2}$'))
_vmin9e, _vmax9e = np.nanpercentile(fal9e, [2, 98])
_norm9e = mcolors.LogNorm(vmin=_vmin9e, vmax=_vmax9e)
fig, axes = plt.subplots(2, 2, figsize=(11.5, 9), sharex=True)
for ax, (col, lab) in zip(axes.ravel(), _pan9e):
    y = np.asarray(T9E[col], float)
    for flag, mk, siz in ((False, 'o', 16), (True, '^', 22)):
        m = (agn9e == flag) & np.isfinite(y) & np.isfinite(chi9e) \
            & np.isfinite(sal9e)
        sc9e = ax.scatter(chi9e[m], y[m], c=fal9e[m], s=siz, marker=mk,
                          cmap='viridis', norm=_norm9e,
                          alpha=0.75, linewidths=0)
    ax.set_xscale('log')
    ax.set_ylabel(lab)
    ax.grid(alpha=0.2, lw=0.5)
    m = np.isfinite(y) & np.isfinite(chi9e)
    if spearmanr is not None and m.sum() > 3:
        rho, p = spearmanr(np.log10(chi9e[m]), y[m])
        ax.set_title(rf'Spearman $\rho={rho:+.2f}$ (p={p:.1e}, N={m.sum()})',
                     fontsize=10)
for ax in axes[1]:
    ax.set_xlabel(r'CIGALE reduced $\chi^2$ (mock dust_on, 19 bands)')
axes[0, 0].scatter([], [], marker='o', color='0.5', label='non-AGN host')
axes[0, 0].scatter([], [], marker='^', color='0.5', label='AGN host')
axes[0, 0].legend(frameon=False, fontsize=9, loc='lower left')
cb9e = fig.colorbar(sc9e, ax=axes, pad=0.02, shrink=0.92)
cb9e.set_label(r'$S_{\rm 1.3\,mm}^{\rm mock}$ [$\mu$Jy]  '
               r'(ALMA band 6, dust_on findx0)')
cb9e.ax.yaxis.set_major_formatter(_fmt9e)
cb9e.ax.yaxis.set_minor_formatter(mticker.NullFormatter())
fig.suptitle('is a poor CIGALE fit just a dust-poor analogue?', y=0.99)
fig.savefig(os.path.join(FIGDIR, 'chi2_vs_dust_content.png'), dpi=180,
            bbox_inches='tight')
plt.show()

# ── figure 2: the causal middle link — chi2 vs the sub-mm flux itself ──
ds9e = np.asarray(T9E['ds'], float)
fig, ax = plt.subplots(figsize=(6.8, 5.4))
m = np.isfinite(sal9e) & np.isfinite(chi9e) & np.isfinite(ds9e)
sc9e = ax.scatter(fal9e[m], chi9e[m], c=ds9e[m], s=16, cmap='plasma',
                  vmin=np.nanpercentile(ds9e, 2),
                  vmax=np.nanpercentile(ds9e, 98), alpha=0.75, linewidths=0)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set(xlabel=r'$S_{\rm 1.3\,mm}^{\rm mock}$ [$\mu$Jy] (ALMA band 6)',
       ylabel=r'CIGALE reduced $\chi^2$')
ax.xaxis.set_major_formatter(_fmt9e)
ax.xaxis.set_minor_formatter(mticker.NullFormatter())
ax.grid(alpha=0.2, lw=0.5)
fig.colorbar(sc9e, ax=ax, label=r'$\log\,M_{\rm dust}/M_\star$')
if spearmanr is not None and m.sum() > 3:
    rho, p = spearmanr(sal9e[m], np.log10(chi9e[m]))
    ax.set_title(rf'Spearman $\rho={rho:+.2f}$ (p={p:.1e}, N={m.sum()})',
                 fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'chi2_vs_alma_flux.png'), dpi=180)
plt.show()

# ── the theory, in one number: chi2 in the dust-poor vs dust-rich quartile ──
mq = np.isfinite(ds9e) & np.isfinite(chi9e)
q1, q3 = np.nanpercentile(ds9e[mq], [25, 75])
lo, hi = chi9e[mq & (ds9e <= q1)], chi9e[mq & (ds9e >= q3)]
print(f'log Md/M* quartiles: <= {q1:.2f} (dust-poor) | >= {q3:.2f} (dust-rich)')
print(f'median reduced chi2: dust-poor {np.median(lo):.2f} (N={lo.size}) vs '
      f'dust-rich {np.median(hi):.2f} (N={hi.size})')

# ── findx0 vs sightline-mean: how much of the chi2 tail is anisotropy? ──
# CIGALE's energy balance is isotropic; a nearly unattenuated sightline over
# quasi-isotropic dust emission is unfittable by construction. The smean runs
# fit the 4-sightline-mean photometry with the SAME grids: objects whose chi2
# collapses there are anisotropy-limited, not model-limited.
_chi9e_ = lambda r_: float(np.ma.filled(r_['best.reduced_chi_square'], np.nan))
CFITM = {}
_smdirs = (sorted(glob.glob(os.path.join(CIGRUNS,
                                         'dust_on_smean_snap*_Zs*_sfhfile3')))
           or sorted(glob.glob(os.path.join(CIGRUNS,
                                            'dust_on_smean_snap*_Zs*_sfhfile2'))))
for rd in _smdirs:
    f_ = os.path.join(rd, 'out', 'results.fits')
    if os.path.exists(f_):
        for r in Table.read(f_):
            CFITM[str(r['id'])] = r
_ids_sm = [i_ for i_ in CFIT if i_ in CFITM]
_c0 = np.array([_chi9e_(CFIT[i_]) for i_ in _ids_sm])
_cm = np.array([_chi9e_(CFITM[i_]) for i_ in _ids_sm])
_mfin = np.isfinite(_c0) & np.isfinite(_cm)
if _mfin.any():
    _agn_sm = np.array([_agn9e.get((int(i_[4:7]), int(i_.split('_gal')[1])),
                                   False) for i_ in _ids_sm])
    fig, ax = plt.subplots(figsize=(6.4, 5.8))
    for flag, mk, lab in ((False, 'o', 'non-AGN host'),
                          (True, '^', 'AGN host')):
        m_ = _mfin & (_agn_sm == flag)
        ax.scatter(_c0[m_], _cm[m_], s=22, marker=mk, alpha=0.8,
                   linewidths=0, label=lab)
    _lo = 0.7 * min(_c0[_mfin].min(), _cm[_mfin].min())
    _hi = 1.4 * max(_c0[_mfin].max(), _cm[_mfin].max())
    ax.plot([_lo, _hi], [_lo, _hi], color='0.6', lw=1, zorder=0)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(_lo, _hi)
    ax.set_ylim(_lo, _hi)
    ax.set(xlabel=r'reduced $\chi^2$ — findx0 (single sightline)',
           ylabel=r'reduced $\chi^2$ — sightline mean (isotropized)')
    ax.grid(alpha=0.2, lw=0.5)
    ax.legend(frameon=False, fontsize=9)
    ax.set_title('below the 1:1 line = anisotropy-limited fits', fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, 'chi2_findx0_vs_smean.png'), dpi=180)
    plt.show()
    _tail = _mfin & (_c0 > 5.0)
    print(f'[smean] {int(_mfin.sum())} paired fits | median chi2 '
          f'{np.median(_c0[_mfin]):.2f} (findx0) -> '
          f'{np.median(_cm[_mfin]):.2f} (smean) | of {int(_tail.sum())} '
          f'tail objects (findx0 chi2>5), '
          f'{int((_cm[_tail] < 2.0).sum())} drop below 2 when isotropized')
else:
    print('[smean] no dust_on_smean *_sfhfile2 results yet — anisotropy '
          'diagnostic skipped (run 9a+9b, submit, then re-run this cell)')